# Face Recognition App Development

In [7]:
pip install face-recognition

  Using cached face_recognition-1.3.0-py2.py3-none-any.whl.metadata (21 kB)
  Using cached face_recognition_models-0.3.0-py2.py3-none-any.whl
  Using cached dlib-20.0.1.tar.gz (3.3 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
Using cached face_recognition-1.3.0-py2.py3-none-any.whl (15 kB)
  Created wheel for dlib: filename=dlib-20.0.1-cp311-cp311-win_amd64.whl size=2731653 sha256=0b48038a12ed76c48aa04688c95647d124d2f83685c6e0276396453e329d2f48
  Stored in directory: c:\users\ahmedjaber\appdata\local\pip\cache\wheels\56\a8\fe\cdaa347540c83e794da01a64cf2d3b26b86effba19deb2e24d
Successfully built dlib

   ---------------------------------------- 0/3 [face-recognition-models]
   --------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import cv2
import os
import numpy as np
import random

# Use the same path as your previous cells
SAMPLES_DIR = r"d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition\live_samples"

def augment_image(img):
    augmentations = []
    
    # 1. Original
    augmentations.append(img)
    
    # 2. Horizontal Flip
    augmentations.append(cv2.flip(img, 1))
    
    # 3. Brightness variations
    bright = cv2.convertScaleAbs(img, alpha=1.2, beta=30)
    dark = cv2.convertScaleAbs(img, alpha=0.8, beta=-30)
    augmentations.extend([bright, dark])
    
    # 4. Small Rotations
    rows, cols = img.shape
    for angle in [-10, 10]:
        M = cv2.getRotationMatrix2D((cols/2, rows/2), angle, 1)
        rotated = cv2.warpAffine(img, M, (cols, rows))
        augmentations.append(rotated)
        
    # 5. Slight Blur (simulates motion)
    augmentations.append(cv2.GaussianBlur(img, (3, 3), 0))
    
    return augmentations

print("[STARTING AUGMENTATION] This will turn 20 images into ~140 per person...")

for person_name in os.listdir(SAMPLES_DIR):
    person_path = os.path.join(SAMPLES_DIR, person_name)
    if not os.path.isdir(person_path): continue
    
    images = [f for f in os.listdir(person_path) if f.endswith(".jpg") and "aug" not in f]
    
    for img_name in images:
        img_ptr = cv2.imread(os.path.join(person_path, img_name), cv2.IMREAD_GRAYSCALE)
        if img_ptr is None: continue
        
        aug_versions = augment_image(img_ptr)
        for i, aug_img in enumerate(aug_versions):
            # Save augmented images with a prefix so we don't re-augment them
            new_name = f"aug_{i}_{img_name}"
            cv2.imwrite(os.path.join(person_path, new_name), aug_img)

print("[FINISHED] Your dataset is now much larger. Now run Cell 2 (Training).")

[STARTING AUGMENTATION] This will turn 20 images into ~140 per person...
[FINISHED] Your dataset is now much larger. Now run Cell 2 (Training).


In [ ]:
# ============================================================
#  CELL 1 — Capture 20 live webcam samples per person
# ============================================================
import os, time
import numpy as np
import cv2

PROJECT_DIR  = r"d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition"
SAMPLES_DIR  = os.path.join(PROJECT_DIR, "live_samples")
CASCADE_PATH = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
os.makedirs(SAMPLES_DIR, exist_ok=True)

detector = cv2.CascadeClassifier(CASCADE_PATH)

# ── Edit this list — one entry per person to enroll ──────────
PERSONS = ["Ahmad Imad", "Ahmed Ashraf", "Ahmed Jaber", "Ahmed Kamel"]
SAMPLES_PER_PERSON = 20

def capture_person(name, n):
    save_dir = os.path.join(SAMPLES_DIR, name)
    os.makedirs(save_dir, exist_ok=True)

    cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
    if not cap.isOpened():
        cap = cv2.VideoCapture(0)

    cap.set(cv2.CAP_PROP_FRAME_WIDTH,  1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

    # warm-up
    for _ in range(15):
        cap.read()

    count   = 0
    print(f"\n[CAPTURING] '{name}' — look at the camera.")
    print(f"  Press SPACE to capture a sample ({n} needed) | Q to skip this person\n")

    while count < n:
        ret, frame = cap.read()
        if not ret:
            continue

        gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray  = cv2.equalizeHist(gray)
        faces = detector.detectMultiScale(
            gray, scaleFactor=1.05, minNeighbors=3, minSize=(50, 50)
        )

        display = frame.copy()
        face_found = False

        for (x, y, w, h) in faces:
            face_found = True
            cv2.rectangle(display, (x, y), (x+w, y+h), (0, 255, 0), 2)

        status = f"'{name}'  captured: {count}/{n}  |  SPACE=capture  Q=skip"
        hint   = "Face detected — ready!" if face_found else "No face detected — move closer"
        color  = (0, 200, 0) if face_found else (0, 0, 220)

        cv2.putText(display, status, (10, 30),
                    cv2.FONT_HERSHEY_DUPLEX, 0.7, (255, 255, 255), 1)
        cv2.putText(display, hint, (10, 65),
                    cv2.FONT_HERSHEY_DUPLEX, 0.65, color, 1)

        cv2.imshow(f"Enrolling: {name}  |  SPACE=capture  Q=skip", display)
        key = cv2.waitKey(1) & 0xFF

        if key == ord(" ") and face_found:
            # save the face crop
            x, y, w, h = faces[0]
            face_crop  = cv2.resize(gray[y:y+h, x:x+w], (200, 200))
            fname      = os.path.join(save_dir, f"{count:03d}.jpg")
            cv2.imwrite(fname, face_crop)
            count += 1
            print(f"  [SAVED] {fname}")
            time.sleep(0.3)   # small pause between captures

        elif key == ord("q"):
            print(f"  [SKIPPED] '{name}'")
            break

    cap.release()
    cv2.destroyAllWindows()
    print(f"  [DONE] '{name}' — {count} samples saved to {save_dir}")
    return count

# ── Run capture for each person ───────────────────────────────
for person in PERSONS:
    existing = [f for f in os.listdir(os.path.join(SAMPLES_DIR, person))
                if f.endswith(".jpg")] if os.path.isdir(
                    os.path.join(SAMPLES_DIR, person)) else []
    if len(existing) >= SAMPLES_PER_PERSON:
        print(f"[SKIP] '{person}' already has {len(existing)} samples.")
        continue
    capture_person(person, SAMPLES_PER_PERSON)

print("\n[ALL DONE] Run Cell 2 to train and start recognition.")


# ============================================================
#  CELL 2 — Train on live samples + run camera
# ============================================================
import json
import numpy as np
import cv2
import os
import time

PROJECT_DIR  = r"d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition"
SAMPLES_DIR  = os.path.join(PROJECT_DIR, "live_samples")
CASCADE_PATH = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
detector     = cv2.CascadeClassifier(CASCADE_PATH)

# ── Load samples ──────────────────────────────────────────────
faces_train  = []
labels_train = []
label_map    = {}
label_id     = 0

for person_name in sorted(os.listdir(SAMPLES_DIR)):
    person_dir = os.path.join(SAMPLES_DIR, person_name)
    if not os.path.isdir(person_dir):
        continue

    imgs = [f for f in os.listdir(person_dir) if f.endswith(".jpg")]
    if len(imgs) == 0:
        continue

    label_map[label_id] = person_name
    count = 0
    for fname in imgs:
        img = cv2.imread(os.path.join(person_dir, fname), cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        img = cv2.resize(img, (200, 200))
        faces_train.append(img)
        labels_train.append(label_id)
        count += 1

    print(f"  [OK] '{person_name}' → id={label_id}  samples={count}")
    label_id += 1

print(f"\n[INFO] Total samples: {len(faces_train)} | Persons: {len(label_map)}")

# ── Train ─────────────────────────────────────────────────────
recognizer = cv2.face.LBPHFaceRecognizer_create(
    radius=2, neighbors=16, grid_x=8, grid_y=8
)
recognizer.train(faces_train, np.array(labels_train))
print(f"[INFO] Training complete. Enrolled: {list(label_map.values())}")

# ── Live recognition ──────────────────────────────────────────
# Start with 100 — watch printed scores and adjust
THRESHOLD = 100

def process_frame(frame):
    gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray  = cv2.equalizeHist(gray)
    faces = detector.detectMultiScale(
        gray, scaleFactor=1.05, minNeighbors=3,
        minSize=(50, 50), flags=cv2.CASCADE_SCALE_IMAGE
    )
    for (x, y, w, h) in faces:
        face_crop       = cv2.resize(gray[y:y+h, x:x+w], (200, 200))
        label_id, score = recognizer.predict(face_crop)
        print(f"  predict → '{label_map.get(label_id,'?')}'  score={score:.1f}")

        if score < THRESHOLD:
            name  = label_map.get(label_id, "Unknown")
            conf  = round(max(0, 100 - score), 1)
            color = (0, 200, 0)
        else:
            name  = "Unknown"
            conf  = 0.0
            color = (0, 0, 220)

        label = f"{name}  {conf}%" if conf else name
        cv2.rectangle(frame, (x, y),   (x+w, y+h),    color, 2)
        cv2.rectangle(frame, (x, y+h), (x+w, y+h+30), color, cv2.FILLED)
        cv2.putText(frame, label, (x+4, y+h+22),
                    cv2.FONT_HERSHEY_DUPLEX, 0.6, (255, 255, 255), 1)
    return frame

cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
if not cap.isOpened():
    cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("[ERROR] Cannot open camera.")
else:
    cap.set(cv2.CAP_PROP_FRAME_WIDTH,  1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    print("\n[INFO] Camera open. Press Q to quit.\n")
    for _ in range(15):
        cap.read()
    while True:
        ret, frame = cap.read()
        if not ret:
            time.sleep(0.03)
            continue
        cv2.imshow("Face Recognition  |  Q to quit", process_frame(frame))
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
    cap.release()
    cv2.destroyAllWindows()
    print("[INFO] Done.")



[CAPTURING] 'Ahmad Imad' — look at the camera.
  Press SPACE to capture a sample (20 needed) | Q to skip this person  [SAVED] d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition\live_samples\Ahmad Imad\000.jpg  [SAVED] d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition\live_samples\Ahmad Imad\001.jpg  [SAVED] d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition\live_samples\Ahmad Imad\002.jpg  [SAVED] d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition\live_samples\Ahmad Imad\003.jpg  [SAVED] d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition\live_samples\Ahmad Imad\004.jpg  [SAVED] d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition\live_samples\Ahmad Imad\005.jpg  [SAVED] d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition\live_samples\Ahmad Imad\006.jpg  [SAVED] d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition\live_samples\Ahmad Imad\007.jpg  [SAVED] d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition\live_samples\Ahmad Imad\008.jpg  [SAVED] d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition\live_samples\Ahmad Imad\009.jpg  [SAVED] d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition\live_samples\Ahmad Imad\010.jpg  [SAVED] d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition\live_samples\Ahmad Imad\011.jpg  [SAVED] d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition\live_samples\Ahmad Imad\012.jpg  [SAVED] d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition\live_samples\Ahmad Imad\013.jpg  [SAVED] d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition\live_samples\Ahmad Imad\014.jpg  [SAVED] d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition\live_samples\Ahmad Imad\015.jpg  [SAVED] d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition\live_samples\Ahmad Imad\016.jpg  [SAVED] d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition\live_samples\Ahmad Imad\017.jpg  [SAVED] d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition\live_samples\Ahmad Imad\018.jpg  [SAVED] d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition\live_samples\Ahmad Imad\019.jpg  [DONE] 'Ahmad Imad' — 20 samples saved to d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition\live_samples\Ahmad Imad
...
  predict → 'Ahmed Ashraf'  score=104.7  predict → 'Ahmad Imad'  score=157.6  predict → 'Ahmed Kamel'  score=120.7[INFO] Done.
Output is truncated. View as a scrollable element or open in a text editor. Adjust cell output settings...

we added 20 pics of everyone using the cam roll but the model still fails to recognise them , can we make augmentation for the 20 to be 200 or what to do ? wdyt

[INFO] Loading images and preparing training data...
  [OK] 'Ahmad Imad' → id=0 | Total samples (Original+Aug): 160
  [OK] 'Ahmed Ashraf' → id=1 | Total samples (Original+Aug): 160
  [OK] 'Ahmed Jaber' → id=2 | Total samples (Original+Aug): 160
  [OK] 'Ahmed Kamel' → id=3 | Total samples (Original+Aug): 160

[INFO] Total dataset size: 640 images.


AttributeError: module 'cv2.face' has no attribute 'LBPHFaceRecognizer_create'

In [8]:
pip install opencv-python

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
# ============================================================
#  CELL — Fix seeding with aggressive detection + debug output
# ============================================================
import os, json
import numpy as np
import cv2

# ── Paths ────────────────────────────────────────────────────
DB_PATH   = r"database.json"
KNOWN_DIR = r"known_faces"

CASCADE_PATH = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
detector     = cv2.CascadeClassifier(CASCADE_PATH)

# ── Embedding ────────────────────────────────────────────────
def get_embedding(face_bgr):
    face = cv2.resize(face_bgr, (64, 64))
    gray = cv2.cvtColor(face, cv2.COLOR_BGR2GRAY)
    hog  = []
    for r in range(8):
        for c in range(8):
            block = gray[r*8:(r+1)*8, c*8:(c+1)*8]
            hist, _ = np.histogram(block.flatten(), bins=8, range=(0, 256))
            hog.extend(hist)
    emb = np.array(hog, dtype=np.float32)
    emb /= (np.linalg.norm(emb) + 1e-9)
    return emb

# ── Try to detect with progressively looser params ───────────
def detect_face(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.equalizeHist(gray)          # boost contrast

    attempts = [
        dict(scaleFactor=1.05, minNeighbors=3, minSize=(30,  30)),
        dict(scaleFactor=1.05, minNeighbors=2, minSize=(20,  20)),
        dict(scaleFactor=1.03, minNeighbors=1, minSize=(20,  20)),
    ]
    for params in attempts:
        faces = detector.detectMultiScale(gray, **params)
        if len(faces) > 0:
            # pick largest face
            areas = [w*h for (x,y,w,h) in faces]
            return faces[int(np.argmax(areas))]
    return None

# ── Seed loop ────────────────────────────────────────────────
persons = []
files   = [f for f in os.listdir(KNOWN_DIR)
           if f.lower().endswith((".jpg", ".jpeg", ".png"))]

print(f"[INFO] Found {len(files)} image(s) in {KNOWN_DIR}\n")

for fname in files:
    fpath = os.path.join(KNOWN_DIR, fname)
    name  = os.path.splitext(fname)[0]
    img   = cv2.imread(fpath)

    if img is None:
        print(f"  [ERROR] Cannot read {fname} — skipping.")
        continue

    h, w  = img.shape[:2]
    print(f"  [{fname}]  size={w}x{h}")

    # resize very large images to speed up detection
    scale = 1.0
    if max(h, w) > 1200:
        scale = 1200 / max(h, w)
        img   = cv2.resize(img, (int(w*scale), int(h*scale)))
        print(f"    → resized to {img.shape[1]}x{img.shape[0]}")

    face = detect_face(img)

    if face is None:
        # ── fallback: use centre 60% of image as face region ──
        print(f"    [WARN] No face detected automatically.")
        print(f"    [FALLBACK] Using centre crop as face region.")
        ih, iw = img.shape[:2]
        margin = 0.20
        x = int(iw * margin)
        y = int(ih * margin)
        fw = int(iw * (1 - 2*margin))
        fh = int(ih * (1 - 2*margin))
        face = (x, y, fw, fh)

    x, y, fw, fh = face
    crop = img[y:y+fh, x:x+fw]
    emb  = get_embedding(crop)

    persons.append({"name": name, "embedding": emb.tolist()})
    print(f"    [OK] '{name}' enrolled.  face_region=({x},{y},{fw},{fh})")

    # save debug image so you can verify the crop looks correct
    debug_dir  = os.path.join(KNOWN_DIR, "debug")
    os.makedirs(debug_dir, exist_ok=True)
    debug_img  = img.copy()
    cv2.rectangle(debug_img, (x, y), (x+fw, y+fh), (0, 200, 0), 2)
    cv2.putText(debug_img, name, (x, y-8),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 200, 0), 2)
    cv2.imwrite(os.path.join(debug_dir, fname), debug_img)

print(f"\n[INFO] Enrolled: {len(persons)}  |  Skipped: {len(files)-len(persons)}")

# ── Save DB ──────────────────────────────────────────────────
with open(DB_PATH, "w") as f:
    json.dump({"persons": persons}, f, indent=2)

print(f"[INFO] Saved to {DB_PATH}")
print(f"\n[INFO] Debug crops saved to {KNOWN_DIR}\\debug\\")
print(        "       Open those images to confirm the face region looks correct.")
print(f"\nEnrolled names: {[p['name'] for p in persons]}")

[INFO] Found 5 image(s) in known_faces

  [Ahmad Imad.jpeg]  size=1599x1599
    → resized to 1200x1200
    [OK] 'Ahmad Imad' enrolled.  face_region=(308,45,722,722)
  [Ahmed Ashraf.jpeg]  size=968x1265
    → resized to 918x1200
    [OK] 'Ahmed Ashraf' enrolled.  face_region=(182,212,559,559)
  [Ahmed Jaber.JPG]  size=1063x1417
    → resized to 900x1200
    [OK] 'Ahmed Jaber' enrolled.  face_region=(262,670,475,475)
  [Ahmed Kamel.jpeg]  size=251x310
    [OK] 'Ahmed Kamel' enrolled.  face_region=(36,43,154,154)
  [WIN_20260508_12_10_55_Pro.jpg]  size=1280x720
    → resized to 1200x675
    [OK] 'WIN_20260508_12_10_55_Pro' enrolled.  face_region=(553,285,262,262)

[INFO] Enrolled: 5  |  Skipped: 0
[INFO] Saved to database.json

[INFO] Debug crops saved to known_faces\debug\
       Open those images to confirm the face region looks correct.

Enrolled names: ['Ahmad Imad', 'Ahmed Ashraf', 'Ahmed Jaber', 'Ahmed Kamel', 'WIN_20260508_12_10_55_Pro']


In [10]:
pip install opencv-contrib-python

   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   --------- ------------------------------ 3.1/12.6 MB 16.8 MB/s eta 0:00:01
   ------------------------- -------------- 8.1/12.6 MB 24.0 MB/s eta 0:00:01
   ------------------------- -------------- 8.1/12.6 MB 24.0 MB/s eta 0:00:01
   ------------------------------ --------- 9.7/12.6 MB 12.3 MB/s eta 0:00:01
   -------------------------------------- - 12.1/12.6 MB 11.6 MB/s eta 0:00:01
   ---------------------------------------- 12.6/12.6 MB 11.3 MB/s  0:00:01
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
deepface 0.0.99 requires opencv-python>=4.5.5.64, which is not installed.
retina-face 0.0.17 requires opencv-python>=3.4.4, which is not installed.
facenet-pytorch 2.6.0 requires numpy<2.0.0,>=1.24.0, but you have numpy 2.4.4 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
# ============================================================
#  CELL 1 — Reseed with augmentation (10 samples per photo)
# ============================================================
import os, json
import numpy as np
import cv2

PROJECT_DIR  = r"d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition"
KNOWN_DIR    = os.path.join(PROJECT_DIR, "known_faces")
MODEL_PATH   = os.path.join(PROJECT_DIR, "lbph_model.yml")
LABELS_PATH  = os.path.join(PROJECT_DIR, "lbph_labels.json")
CASCADE_PATH = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"

detector = cv2.CascadeClassifier(CASCADE_PATH)

# ── Augment a single face crop into N variations ─────────────
def augment(face_gray, n=20):
    samples = [face_gray]
    h, w    = face_gray.shape
    for i in range(n - 1):
        aug = face_gray.copy()
        # random brightness
        delta = np.random.randint(-40, 40)
        aug   = np.clip(aug.astype(np.int32) + delta, 0, 255).astype(np.uint8)
        # random flip
        if np.random.rand() > 0.5:
            aug = cv2.flip(aug, 1)
        # random small rotation
        angle = np.random.uniform(-15, 15)
        M     = cv2.getRotationMatrix2D((w//2, h//2), angle, 1.0)
        aug   = cv2.warpAffine(aug, M, (w, h))
        # random slight scale crop
        scale = np.random.uniform(0.85, 1.0)
        crop_w, crop_h = int(w*scale), int(h*scale)
        ox = np.random.randint(0, w - crop_w + 1)
        oy = np.random.randint(0, h - crop_h + 1)
        aug = cv2.resize(aug[oy:oy+crop_h, ox:ox+crop_w], (w, h))

        samples.append(aug)
    return samples

def detect_face(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.equalizeHist(gray)
    for params in [
        dict(scaleFactor=1.05, minNeighbors=3, minSize=(30, 30)),
        dict(scaleFactor=1.05, minNeighbors=2, minSize=(20, 20)),
        dict(scaleFactor=1.03, minNeighbors=1, minSize=(20, 20)),
    ]:
        faces = detector.detectMultiScale(gray, **params)
        if len(faces) > 0:
            areas = [w*h for (x,y,w,h) in faces]
            return faces[int(np.argmax(areas))], gray
    ih, iw = gray.shape
    m = 0.20
    return (int(iw*m), int(ih*m), int(iw*0.6), int(ih*0.6)), gray

# ── Collect samples ──────────────────────────────────────────
faces_gray = []
labels     = []
label_map  = {}
label_id   = 0

files = [f for f in os.listdir(KNOWN_DIR)
         if f.lower().endswith((".jpg", ".jpeg", ".png"))
         and "debug" not in f.lower()]

# skip webcam snapshot — only use proper portrait photos
SKIP = ["win_"]   # lowercase prefixes to skip
files = [f for f in files
         if not any(f.lower().startswith(s) for s in SKIP)]

print(f"[INFO] Training on {len(files)} image(s)\n")

for fname in files:
    name = os.path.splitext(fname)[0]
    img  = cv2.imread(os.path.join(KNOWN_DIR, fname))
    if img is None:
        continue

    h, w = img.shape[:2]
    if max(h, w) > 1200:
        scale = 1200 / max(h, w)
        img   = cv2.resize(img, (int(w*scale), int(h*scale)))

    (x, y, fw, fh), gray_eq = detect_face(img)
    face_crop = cv2.resize(gray_eq[y:y+fh, x:x+fw], (200, 200))

    if name not in label_map.values():
        label_map[label_id] = name
        current_id = label_id
        label_id += 1
    else:
        current_id = [k for k,v in label_map.items() if v == name][0]

    samples = augment(face_crop, n=20)
    faces_gray.extend(samples)
    labels.extend([current_id] * len(samples))
    print(f"  [OK] '{name}' → id={current_id}  samples={len(samples)}")

# ── Train ─────────────────────────────────────────────────────
recognizer = cv2.face.LBPHFaceRecognizer_create(
    radius=2, neighbors=16, grid_x=8, grid_y=8
)
recognizer.train(faces_gray, np.array(labels))
recognizer.save(MODEL_PATH)

with open(LABELS_PATH, "w") as f:
    json.dump(label_map, f)

print(f"\n[INFO] Trained {len(faces_gray)} samples across {len(label_map)} person(s)")
print(f"[INFO] Model saved → {MODEL_PATH}")


# ============================================================
#  CELL 2 — Live camera with tuned threshold
# ============================================================
import time

recognizer = cv2.face.LBPHFaceRecognizer_create()
recognizer.read(MODEL_PATH)

with open(LABELS_PATH) as f:
    label_map = {int(k): v for k, v in json.load(f).items()}

print(f"[INFO] Loaded model. Enrolled: {list(label_map.values())}")

# With augmentation scores drop to ~60-90 for matches
# Adjust this if still showing Unknown (raise it) or wrong name (lower it)
THRESHOLD = 110

def process_frame(frame):
    gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray  = cv2.equalizeHist(gray)
    faces = detector.detectMultiScale(
        gray, scaleFactor=1.05, minNeighbors=3,
        minSize=(50, 50), flags=cv2.CASCADE_SCALE_IMAGE
    )

    for (x, y, w, h) in faces:
        face_crop        = cv2.resize(gray[y:y+h, x:x+w], (200, 200))
        label_id, score  = recognizer.predict(face_crop)
        print(f"  predict → '{label_map.get(label_id,'?')}'  score={score:.1f}")

        if score < THRESHOLD:
            name  = label_map.get(label_id, "Unknown")
            conf  = round(max(0, 100 - score), 1)
            color = (0, 200, 0)
        else:
            name  = "Unknown"
            conf  = 0.0
            color = (0, 0, 220)

        label = f"{name}  {conf}%" if conf else name
        cv2.rectangle(frame, (x, y),       (x+w, y+h),    color, 2)
        cv2.rectangle(frame, (x, y+h),     (x+w, y+h+30), color, cv2.FILLED)
        cv2.putText(frame, label, (x+4, y+h+22),
                    cv2.FONT_HERSHEY_DUPLEX, 0.6, (255, 255, 255), 1)
    return frame

cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
if not cap.isOpened():
    cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("[ERROR] Cannot open camera.")
else:
    cap.set(cv2.CAP_PROP_FRAME_WIDTH,  1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    print("[INFO] Camera open. Press Q to quit.\n")
    for _ in range(15):
        cap.read()
    while True:
        ret, frame = cap.read()
        if not ret:
            time.sleep(0.03)
            continue
        cv2.imshow("Face Recognition  |  Q to quit", process_frame(frame))
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
    cap.release()
    cv2.destroyAllWindows()
    print("[INFO] Done.")

[INFO] Training on 4 image(s)



error: OpenCV(4.11.0) D:\a\opencv-python\opencv-python\opencv\modules\objdetect\src\cascadedetect.cpp:1689: error: (-215:Assertion failed) !empty() in function 'cv::CascadeClassifier::detectMultiScale'


In [10]:
# ============================================================
#  CELL 1 — Capture 20 live webcam samples per person
# ============================================================
import os, time
import numpy as np
import cv2

PROJECT_DIR  = r"d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition"
SAMPLES_DIR  = os.path.join(PROJECT_DIR, "live_samples")
CASCADE_PATH = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
os.makedirs(SAMPLES_DIR, exist_ok=True)

detector = cv2.CascadeClassifier(CASCADE_PATH)

# ── Edit this list — one entry per person to enroll ──────────
PERSONS = ["Ahmad Imad", "Ahmed Ashraf", "Ahmed Jaber", "Ahmed Kamel"]
SAMPLES_PER_PERSON = 20

def capture_person(name, n):
    save_dir = os.path.join(SAMPLES_DIR, name)
    os.makedirs(save_dir, exist_ok=True)

    cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
    if not cap.isOpened():
        cap = cv2.VideoCapture(0)

    cap.set(cv2.CAP_PROP_FRAME_WIDTH,  1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

    # warm-up
    for _ in range(15):
        cap.read()

    count   = 0
    print(f"\n[CAPTURING] '{name}' — look at the camera.")
    print(f"  Press SPACE to capture a sample ({n} needed) | Q to skip this person\n")

    while count < n:
        ret, frame = cap.read()
        if not ret:
            continue

        gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray  = cv2.equalizeHist(gray)
        faces = detector.detectMultiScale(
            gray, scaleFactor=1.05, minNeighbors=3, minSize=(50, 50)
        )

        display = frame.copy()
        face_found = False

        for (x, y, w, h) in faces:
            face_found = True
            cv2.rectangle(display, (x, y), (x+w, y+h), (0, 255, 0), 2)

        status = f"'{name}'  captured: {count}/{n}  |  SPACE=capture  Q=skip"
        hint   = "Face detected — ready!" if face_found else "No face detected — move closer"
        color  = (0, 200, 0) if face_found else (0, 0, 220)

        cv2.putText(display, status, (10, 30),
                    cv2.FONT_HERSHEY_DUPLEX, 0.7, (255, 255, 255), 1)
        cv2.putText(display, hint, (10, 65),
                    cv2.FONT_HERSHEY_DUPLEX, 0.65, color, 1)

        cv2.imshow(f"Enrolling: {name}  |  SPACE=capture  Q=skip", display)
        key = cv2.waitKey(1) & 0xFF

        if key == ord(" ") and face_found:
            # save the face crop
            x, y, w, h = faces[0]
            face_crop  = cv2.resize(gray[y:y+h, x:x+w], (200, 200))
            fname      = os.path.join(save_dir, f"{count:03d}.jpg")
            cv2.imwrite(fname, face_crop)
            count += 1
            print(f"  [SAVED] {fname}")
            time.sleep(0.3)   # small pause between captures

        elif key == ord("q"):
            print(f"  [SKIPPED] '{name}'")
            break

    cap.release()
    cv2.destroyAllWindows()
    print(f"  [DONE] '{name}' — {count} samples saved to {save_dir}")
    return count

# ── Run capture for each person ───────────────────────────────
for person in PERSONS:
    existing = [f for f in os.listdir(os.path.join(SAMPLES_DIR, person))
                if f.endswith(".jpg")] if os.path.isdir(
                    os.path.join(SAMPLES_DIR, person)) else []
    if len(existing) >= SAMPLES_PER_PERSON:
        print(f"[SKIP] '{person}' already has {len(existing)} samples.")
        continue
    capture_person(person, SAMPLES_PER_PERSON)

print("\n[ALL DONE] Run Cell 2 to train and start recognition.")


# ============================================================
#  CELL 2 — Train on live samples + run camera
# ============================================================
import json
import numpy as np
import cv2
import os
import time

PROJECT_DIR  = r"d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition"
SAMPLES_DIR  = os.path.join(PROJECT_DIR, "live_samples")
CASCADE_PATH = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
detector     = cv2.CascadeClassifier(CASCADE_PATH)

# ── Load samples ──────────────────────────────────────────────
faces_train  = []
labels_train = []
label_map    = {}
label_id     = 0

for person_name in sorted(os.listdir(SAMPLES_DIR)):
    person_dir = os.path.join(SAMPLES_DIR, person_name)
    if not os.path.isdir(person_dir):
        continue

    imgs = [f for f in os.listdir(person_dir) if f.endswith(".jpg")]
    if len(imgs) == 0:
        continue

    label_map[label_id] = person_name
    count = 0
    for fname in imgs:
        img = cv2.imread(os.path.join(person_dir, fname), cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        img = cv2.resize(img, (200, 200))
        faces_train.append(img)
        labels_train.append(label_id)
        count += 1

    print(f"  [OK] '{person_name}' → id={label_id}  samples={count}")
    label_id += 1

print(f"\n[INFO] Total samples: {len(faces_train)} | Persons: {len(label_map)}")

# ── Train ─────────────────────────────────────────────────────
recognizer = cv2.face.LBPHFaceRecognizer_create(
    radius=2, neighbors=16, grid_x=8, grid_y=8
)
recognizer.train(faces_train, np.array(labels_train))
print(f"[INFO] Training complete. Enrolled: {list(label_map.values())}")

# ── Live recognition ──────────────────────────────────────────
# Start with 100 — watch printed scores and adjust
THRESHOLD = 100

def process_frame(frame):
    gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray  = cv2.equalizeHist(gray)
    faces = detector.detectMultiScale(
        gray, scaleFactor=1.05, minNeighbors=3,
        minSize=(50, 50), flags=cv2.CASCADE_SCALE_IMAGE
    )
    for (x, y, w, h) in faces:
        face_crop       = cv2.resize(gray[y:y+h, x:x+w], (200, 200))
        label_id, score = recognizer.predict(face_crop)
        print(f"  predict → '{label_map.get(label_id,'?')}'  score={score:.1f}")

        if score < THRESHOLD:
            name  = label_map.get(label_id, "Unknown")
            conf  = round(max(0, 100 - score), 1)
            color = (0, 200, 0)
        else:
            name  = "Unknown"
            conf  = 0.0
            color = (0, 0, 220)

        label = f"{name}  {conf}%" if conf else name
        cv2.rectangle(frame, (x, y),   (x+w, y+h),    color, 2)
        cv2.rectangle(frame, (x, y+h), (x+w, y+h+30), color, cv2.FILLED)
        cv2.putText(frame, label, (x+4, y+h+22),
                    cv2.FONT_HERSHEY_DUPLEX, 0.6, (255, 255, 255), 1)
    return frame

cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
if not cap.isOpened():
    cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("[ERROR] Cannot open camera.")
else:
    cap.set(cv2.CAP_PROP_FRAME_WIDTH,  1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    print("\n[INFO] Camera open. Press Q to quit.\n")
    for _ in range(15):
        cap.read()
    while True:
        ret, frame = cap.read()
        if not ret:
            time.sleep(0.03)
            continue
        cv2.imshow("Face Recognition  |  Q to quit", process_frame(frame))
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
    cap.release()
    cv2.destroyAllWindows()
    print("[INFO] Done.")


[CAPTURING] 'Ahmad Imad' — look at the camera.
  Press SPACE to capture a sample (20 needed) | Q to skip this person

  [SAVED] d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition\live_samples\Ahmad Imad\000.jpg
  [SAVED] d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition\live_samples\Ahmad Imad\001.jpg
  [SAVED] d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition\live_samples\Ahmad Imad\002.jpg
  [SAVED] d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition\live_samples\Ahmad Imad\003.jpg
  [SAVED] d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition\live_samples\Ahmad Imad\004.jpg
  [SAVED] d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition\live_samples\Ahmad Imad\005.jpg
  [SAVED] d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition\live_samples\Ahmad Imad\006.jpg
  [SAVED] d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition\live_samples\Ahmad Imad\007.jpg
  [SAVED] d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition\live_samples\Ahmad Imad\008.jpg
  [SAVED] d:\Self_Study\DEBI\DEBI-H

In [11]:
import cv2
import os
import numpy as np
import random

# Use the same path as your previous cells
SAMPLES_DIR = r"d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition\live_samples"

def augment_image(img):
    augmentations = []
    
    # 1. Original
    augmentations.append(img)
    
    # 2. Horizontal Flip
    augmentations.append(cv2.flip(img, 1))
    
    # 3. Brightness variations
    bright = cv2.convertScaleAbs(img, alpha=1.2, beta=30)
    dark = cv2.convertScaleAbs(img, alpha=0.8, beta=-30)
    augmentations.extend([bright, dark])
    
    # 4. Small Rotations
    rows, cols = img.shape
    for angle in [-10, 10]:
        M = cv2.getRotationMatrix2D((cols/2, rows/2), angle, 1)
        rotated = cv2.warpAffine(img, M, (cols, rows))
        augmentations.append(rotated)
        
    # 5. Slight Blur (simulates motion)
    augmentations.append(cv2.GaussianBlur(img, (3, 3), 0))
    
    return augmentations

print("[STARTING AUGMENTATION] This will turn 20 images into ~140 per person...")

for person_name in os.listdir(SAMPLES_DIR):
    person_path = os.path.join(SAMPLES_DIR, person_name)
    if not os.path.isdir(person_path): continue
    
    images = [f for f in os.listdir(person_path) if f.endswith(".jpg") and "aug" not in f]
    
    for img_name in images:
        img_ptr = cv2.imread(os.path.join(person_path, img_name), cv2.IMREAD_GRAYSCALE)
        if img_ptr is None: continue
        
        aug_versions = augment_image(img_ptr)
        for i, aug_img in enumerate(aug_versions):
            # Save augmented images with a prefix so we don't re-augment them
            new_name = f"aug_{i}_{img_name}"
            cv2.imwrite(os.path.join(person_path, new_name), aug_img)

print("[FINISHED] Your dataset is now much larger. Now run Cell 2 (Training).")

[STARTING AUGMENTATION] This will turn 20 images into ~140 per person...
[FINISHED] Your dataset is now much larger. Now run Cell 2 (Training).


codeeee

In [ ]:
# ============================================================
#  CELL 1 — Run this FIRST, then restart kernel immediately
# ============================================================
import sys, importlib

# purge any cached broken dlib / face_recognition modules
to_delete = [k for k in sys.modules if "dlib" in k or "face_recognition" in k]
for k in to_delete:
    del sys.modules[k]
    print(f"[PURGED] {k}")

print("\n[DONE] Now go to:  Kernel → Restart Kernel  then run Cell 2.")


# ============================================================
#  CELL 2 — Run AFTER kernel restart (fresh session)
# ============================================================
import subprocess, sys

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)

# 1. Uninstall broken versions completely
run(f"{sys.executable} -m pip uninstall dlib face_recognition face_recognition_models -y")

# 2. Install the prebuilt dlib wheel (no compilation, no DLL issues)
run(f"{sys.executable} -m pip install dlib==19.24.2 --only-binary=:all:")

# 3. Install face_recognition on top
run(f"{sys.executable} -m pip install face_recognition==1.3.0")

# 4. Verify
try:
    import dlib
    import face_recognition
    print(f"\n[OK] dlib version      : {dlib.__version__}")
    print(f"[OK] face_recognition  : imported successfully")
except ImportError as e:
    print(f"\n[FAIL] {e}")
    print("\nTry running in terminal instead:")
    print("  pip install dlib==19.24.2 --only-binary=:all:")
    print("  pip install face_recognition==1.3.0")
    print("Then restart VS Code / Jupyter completely and re-run.")


# ============================================================
#  CELL 3 — If Cell 2 dlib install fails (fallback: mediapipe)
#  Only run this if dlib still refuses to install
# ============================================================
run(f"{sys.executable} -m pip install mediapipe opencv-python numpy")

import os, json
import numpy as np
import cv2
import mediapipe as mp
from PIL import Image

# Auto-detect DB_PATH
for candidate in [
    "/content/drive/MyDrive/Hackathon/database.json",
    "/kaggle/working/database.json",
    os.path.join(os.getcwd(), "database.json"),
]:
    if os.path.exists(candidate):
        DB_PATH = candidate
        break
else:
    DB_PATH = os.path.join(os.getcwd(), "database.json")

print(f"[INFO] DB_PATH = {DB_PATH}")

# Load DB
known_embeddings = []
known_names = []
if os.path.exists(DB_PATH):
    with open(DB_PATH) as f:
        data = json.load(f)
    for p in data.get("persons", []):
        known_embeddings.append(np.array(p["embedding"]))
        known_names.append(p["name"])
    print(f"[INFO] Loaded {len(known_names)} face(s): {known_names}")
else:
    print("[WARNING] database.json not found.")

# MediaPipe face detector (replaces dlib)
mp_face = mp.solutions.face_detection
mp_draw = mp.solutions.drawing_utils
detector = mp_face.FaceDetection(model_selection=0, min_detection_confidence=0.6)

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9)

def process_frame_mediapipe(frame_bgr):
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    results = detector.process(rgb)
    h, w = frame_bgr.shape[:2]

    if not results.detections:
        return frame_bgr

    for det in results.detections:
        bb = det.location_data.relative_bounding_box
        x1 = max(0, int(bb.xmin * w))
        y1 = max(0, int(bb.ymin * h))
        x2 = min(w, int((bb.xmin + bb.width) * w))
        y2 = min(h, int((bb.ymin + bb.height) * h))

        # crop face and build a simple mean-color embedding (placeholder)
        face_crop = rgb[y1:y2, x1:x2]
        if face_crop.size == 0:
            continue
        face_resized = cv2.resize(face_crop, (64, 64)).flatten().astype(np.float32)
        face_norm    = face_resized / (np.linalg.norm(face_resized) + 1e-9)

        name       = "Unknown"
        confidence = 0.0
        color      = (0, 0, 220)

        if known_embeddings:
            sims     = [cosine_similarity(face_norm, e) for e in known_embeddings]
            best_idx = int(np.argmax(sims))
            best_sim = sims[best_idx]
            if best_sim > 0.75:
                name       = known_names[best_idx]
                confidence = round(best_sim * 100, 1)
                color      = (0, 200, 0)

        label = f"{name} {confidence}%" if confidence else name
        cv2.rectangle(frame_bgr, (x1, y1), (x2, y2), color, 2)
        cv2.rectangle(frame_bgr, (x1, y2 - 28), (x2, y2), color, cv2.FILLED)
        cv2.putText(frame_bgr, label, (x1 + 4, y2 - 8),
                    cv2.FONT_HERSHEY_DUPLEX, 0.55, (255, 255, 255), 1)

    return frame_bgr

# Camera loop using mediapipe
import time

cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
if not cap.isOpened():
    cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("[ERROR] Cannot open camera. Try index 1 or 2.")
else:
    print("[INFO] Camera opened. Press Q to quit.")
    for _ in range(10):   # warm-up
        cap.read()
    while True:
        ret, frame = cap.read()
        if not ret:
            time.sleep(0.05)
            continue
        annotated = process_frame_mediapipe(frame.copy())
        cv2.imshow("Face Recognition (mediapipe)  |  Q to quit", annotated)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
    cap.release()
    cv2.destroyAllWindows()
    print("[INFO] Done.")

[PURGED] jedi.plugins.stdlib

[DONE] Now go to:  Kernel → Restart Kernel  then run Cell 2.
Found existing installation: dlib 20.0.1
Uninstalling dlib-20.0.1:
  Successfully uninstalled dlib-20.0.1
Found existing installation: face-recognition 1.3.0
Uninstalling face-recognition-1.3.0:
  Successfully uninstalled face-recognition-1.3.0
Found existing installation: face_recognition_models 0.3.0
Uninstalling face_recognition_models-0.3.0:
  Successfully uninstalled face_recognition_models-0.3.0


ERROR: Could not find a version that satisfies the requirement dlib==19.24.2 (from versions: none)

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for dlib==19.24.2



In [15]:
# ============================================================
#  CELL 1 — Install ONLY what's needed (no dlib, no mediapipe)
# ============================================================
import subprocess, sys

subprocess.run(f"{sys.executable} -m pip install opencv-python numpy pillow", shell=True)
print("[DONE] Dependencies ready.")


# ============================================================
#  CELL 2 — Full face recognition using OpenCV only
# ============================================================
import os, json, time
import numpy as np
import cv2

# ── Auto-detect DB_PATH ──────────────────────────────────────
for candidate in [
    "/content/drive/MyDrive/Hackathon/database.json",
    "/kaggle/working/database.json",
    os.path.join(os.getcwd(), "database.json"),
]:
    if os.path.exists(candidate):
        DB_PATH = candidate
        break
else:
    DB_PATH = os.path.join(os.getcwd(), "database.json")

print(f"[INFO] DB_PATH = {DB_PATH}")

# ── Load enrolled faces ──────────────────────────────────────
known_embeddings = []
known_names      = []

if os.path.exists(DB_PATH):
    with open(DB_PATH) as f:
        data = json.load(f)
    for p in data.get("persons", []):
        known_embeddings.append(np.array(p["embedding"], dtype=np.float32))
        known_names.append(p["name"])
    print(f"[INFO] Loaded {len(known_names)} face(s): {known_names}")
else:
    print("[WARNING] database.json not found — all faces will show as Unknown.")

# ── Download OpenCV Haar cascade if missing ──────────────────
CASCADE_PATH = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
if not os.path.exists(CASCADE_PATH):
    import urllib.request
    url = "https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml"
    urllib.request.urlretrieve(url, CASCADE_PATH)
    print(f"[INFO] Downloaded cascade to {CASCADE_PATH}")

detector = cv2.CascadeClassifier(CASCADE_PATH)

# ── Face embedding using pixel histogram (no dlib needed) ────
def get_embedding(face_img):
    """128-dim embedding from resized + normalized face crop."""
    face = cv2.resize(face_img, (64, 64))
    face = cv2.cvtColor(face, cv2.COLOR_BGR2GRAY)
    # 8 histogram bins per 8x8 block → 8*8*8 = 512 → reduce to 128
    hog  = []
    for row in range(8):
        for col in range(8):
            block = face[row*8:(row+1)*8, col*8:(col+1)*8]
            hist, _ = np.histogram(block.flatten(), bins=8, range=(0, 256))
            hog.extend(hist)
    emb = np.array(hog, dtype=np.float32)
    emb /= (np.linalg.norm(emb) + 1e-9)
    return emb

def cosine_sim(a, b):
    return float(np.dot(a, b))   # both already L2-normalised

# ── Re-seed DB with OpenCV embeddings if DB used face_recognition ──
# (face_recognition embeddings are 128-d floats too, but from a different
#  model — we rebuild them from the known_faces/ folder using our embedder)
KNOWN_DIR = os.path.join(os.path.dirname(DB_PATH), "known_faces")
if os.path.isdir(KNOWN_DIR) and len(known_embeddings) == 0:
    print(f"[INFO] Re-seeding from {KNOWN_DIR} using OpenCV embedder...")
    persons = []
    for fname in os.listdir(KNOWN_DIR):
        if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
            continue
        name = os.path.splitext(fname)[0]
        img  = cv2.imread(os.path.join(KNOWN_DIR, fname))
        if img is None:
            continue
        gray  = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        faces = detector.detectMultiScale(gray, 1.1, 5, minSize=(60, 60))
        if len(faces) == 0:
            print(f"  [SKIP] No face found in {fname}")
            continue
        x, y, w, h = faces[0]
        emb = get_embedding(img[y:y+h, x:x+w])
        known_embeddings.append(emb)
        known_names.append(name)
        persons.append({"name": name, "embedding": emb.tolist()})
        print(f"  [OK] {name}")
    with open(DB_PATH, "w") as f:
        json.dump({"persons": persons}, f)
    print(f"[INFO] Saved {len(persons)} face(s) to {DB_PATH}")

elif len(known_embeddings) > 0 and len(known_embeddings[0]) != 512:
    # DB has face_recognition 128-d embeddings → rebuild with our embedder
    print("[INFO] DB embeddings are from face_recognition model.")
    print("       Rebuilding from known_faces/ with OpenCV embedder...")
    if os.path.isdir(KNOWN_DIR):
        persons = []
        known_embeddings.clear()
        known_names.clear()
        for fname in os.listdir(KNOWN_DIR):
            if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
                continue
            name = os.path.splitext(fname)[0]
            img  = cv2.imread(os.path.join(KNOWN_DIR, fname))
            if img is None:
                continue
            gray  = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            faces = detector.detectMultiScale(gray, 1.1, 5, minSize=(60, 60))
            if len(faces) == 0:
                print(f"  [SKIP] No face in {fname}")
                continue
            x, y, w, h = faces[0]
            emb = get_embedding(img[y:y+h, x:x+w])
            known_embeddings.append(emb)
            known_names.append(name)
            persons.append({"name": name, "embedding": emb.tolist()})
            print(f"  [OK] {name}")
        with open(DB_PATH, "w") as f:
            json.dump({"persons": persons}, f)
        print(f"[INFO] Rebuilt DB with {len(persons)} face(s).")

print(f"\n[INFO] Ready — {len(known_names)} enrolled: {known_names}")

# ── process_frame ────────────────────────────────────────────
THRESHOLD = 0.80   # cosine similarity — raise to be stricter

def process_frame(frame):
    gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = detector.detectMultiScale(
        gray, scaleFactor=1.1, minNeighbors=5,
        minSize=(60, 60), flags=cv2.CASCADE_SCALE_IMAGE
    )

    for (x, y, w, h) in faces:
        face_crop = frame[y:y+h, x:x+w]
        emb       = get_embedding(face_crop)

        name       = "Unknown"
        confidence = 0.0
        color      = (0, 0, 220)   # red

        if known_embeddings:
            sims     = [cosine_sim(emb, k) for k in known_embeddings]
            best_idx = int(np.argmax(sims))
            best_sim = sims[best_idx]
            if best_sim >= THRESHOLD:
                name       = known_names[best_idx]
                confidence = round(best_sim * 100, 1)
                color      = (0, 200, 0)   # green

        label = f"{name}  {confidence}%" if confidence else name

        cv2.rectangle(frame, (x, y), (x+w, y+h), color, 2)
        cv2.rectangle(frame, (x, y+h-28), (x+w, y+h), color, cv2.FILLED)
        cv2.putText(frame, label, (x+4, y+h-8),
                    cv2.FONT_HERSHEY_DUPLEX, 0.55, (255, 255, 255), 1)
    return frame

# ── Camera loop ──────────────────────────────────────────────
cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
if not cap.isOpened():
    cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("[ERROR] Cannot open camera index 0. Try changing to 1 or 2.")
else:
    cap.set(cv2.CAP_PROP_FRAME_WIDTH,  1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    print("[INFO] Camera open. Press Q to quit.")
    for _ in range(15):   # warm-up frames
        cap.read()
    while True:
        ret, frame = cap.read()
        if not ret:
            time.sleep(0.03)
            continue
        cv2.imshow("Face Recognition  |  Q to quit", process_frame(frame))
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
    cap.release()
    cv2.destroyAllWindows()
    print("[INFO] Camera released.")

[DONE] Dependencies ready.
[INFO] DB_PATH = d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition\database.json
[INFO] Loaded 5 face(s): ['Ahmad Imad', 'Ahmed Ashraf', 'Ahmed Jaber', 'Ahmed Kamel', 'WIN_20260508_12_10_55_Pro']

[INFO] Ready — 5 enrolled: ['Ahmad Imad', 'Ahmed Ashraf', 'Ahmed Jaber', 'Ahmed Kamel', 'WIN_20260508_12_10_55_Pro']
[INFO] Camera open. Press Q to quit.
[INFO] Camera released.


In [25]:
# ============================================================
#  CELL 0 — Fix cascade path (run this once to verify)
# ============================================================
import cv2, urllib.request, os

# Try built-in path first
CASCADE_PATH = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
print(f"[INFO] Trying: {CASCADE_PATH}")
print(f"[INFO] Exists: {os.path.exists(CASCADE_PATH)}")

# If missing, download it directly
if not os.path.exists(CASCADE_PATH):
    CASCADE_PATH = r"d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition\haarcascade_frontalface_default.xml"
    if not os.path.exists(CASCADE_PATH):
        print("[INFO] Downloading cascade file...")
        urllib.request.urlretrieve(
            "https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml",
            CASCADE_PATH
        )
        print(f"[INFO] Downloaded to {CASCADE_PATH}")

detector = cv2.CascadeClassifier(CASCADE_PATH)
if detector.empty():
    print("[ERROR] Cascade still empty! Check the path above.")
else:
    print(f"[OK] Cascade loaded successfully from:\n     {CASCADE_PATH}")


# ============================================================
#  CELL 1 — Capture 20 live samples for each person
# ============================================================
import os, time
import numpy as np
import cv2

PROJECT_DIR  = r"d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition"
SAMPLES_DIR  = os.path.join(PROJECT_DIR, "live_samples")
os.makedirs(SAMPLES_DIR, exist_ok=True)

# CASCADE_PATH is already set from Cell 0 — reuse it
detector = cv2.CascadeClassifier(CASCADE_PATH)

PERSONS            = ["Ahmad Imad", "Ahmed Jaber"]
SAMPLES_PER_PERSON = 20

def capture_person(name, n):
    save_dir = os.path.join(SAMPLES_DIR, name)
    os.makedirs(save_dir, exist_ok=True)
    for f in os.listdir(save_dir):
        os.remove(os.path.join(save_dir, f))

    cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
    if not cap.isOpened():
        cap = cv2.VideoCapture(0)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH,  1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    for _ in range(15):
        cap.read()

    count = 0
    print(f"\n[CAPTURING] '{name}'")
    print(f"  Press SPACE to capture ({n} needed) | Q to finish early\n")

    while count < n:
        ret, frame = cap.read()
        if not ret:
            continue

        gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray  = cv2.equalizeHist(gray)
        faces = detector.detectMultiScale(
            gray, scaleFactor=1.05, minNeighbors=3, minSize=(50, 50)
        )

        display    = frame.copy()
        face_found = len(faces) > 0
        for (x, y, w, h) in faces:
            cv2.rectangle(display, (x, y), (x+w, y+h), (0, 255, 0), 2)

        hint  = "Face detected — press SPACE" if face_found else "No face — move closer"
        color = (0, 220, 0) if face_found else (0, 0, 220)
        cv2.putText(display, f"'{name}'  {count}/{n}", (10, 35),
                    cv2.FONT_HERSHEY_DUPLEX, 0.8, (255, 255, 255), 1)
        cv2.putText(display, hint, (10, 70),
                    cv2.FONT_HERSHEY_DUPLEX, 0.65, color, 1)

        cv2.imshow(f"Enrolling: {name}  |  SPACE=capture  Q=done", display)
        key = cv2.waitKey(1) & 0xFF

        if key == ord(" ") and face_found:
            x, y, w, h = faces[0]
            crop = cv2.resize(gray[y:y+h, x:x+w], (200, 200))
            cv2.imwrite(os.path.join(save_dir, f"{count:03d}.jpg"), crop)
            count += 1
            print(f"  [SAVED] {count}/{n}")
            time.sleep(0.25)
        elif key == ord("q"):
            print(f"  [STOPPED] '{name}' — {count} samples saved")
            break

    cap.release()
    cv2.destroyAllWindows()
    print(f"  [DONE] '{name}' — {count} samples saved.\n")

for person in PERSONS:
    capture_person(person, SAMPLES_PER_PERSON)

print("[ALL DONE] Run Cell 2 to train and start recognition.")


# ============================================================
#  CELL 2 — Augment + Train + Run
# ============================================================
import numpy as np
import cv2, os, time

PROJECT_DIR = r"d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition"
SAMPLES_DIR = os.path.join(PROJECT_DIR, "live_samples")
PERSONS     = ["Ahmad Imad", "Ahmed Jaber"]

detector = cv2.CascadeClassifier(CASCADE_PATH)

def augment(face_gray, n=15):
    h, w    = face_gray.shape
    samples = [face_gray]
    for _ in range(n - 1):
        aug   = face_gray.copy()
        delta = np.random.randint(-45, 45)
        aug   = np.clip(aug.astype(np.int32) + delta, 0, 255).astype(np.uint8)
        if np.random.rand() > 0.5:
            aug = cv2.flip(aug, 1)
        angle = np.random.uniform(-20, 20)
        M     = cv2.getRotationMatrix2D((w//2, h//2), angle, 1.0)
        aug   = cv2.warpAffine(aug, M, (w, h))
        scale = np.random.uniform(0.80, 1.0)
        cw, ch = int(w*scale), int(h*scale)
        ox = np.random.randint(0, w - cw + 1)
        oy = np.random.randint(0, h - ch + 1)
        aug = cv2.resize(aug[oy:oy+ch, ox:ox+cw], (w, h))
        if np.random.rand() > 0.7:
            aug = cv2.GaussianBlur(aug, (3, 3), 0)
        samples.append(aug)
    return samples

faces_train  = []
labels_train = []
label_map    = {}

print("[INFO] Loading & augmenting samples...\n")

for idx, person_name in enumerate(PERSONS):
    person_dir = os.path.join(SAMPLES_DIR, person_name)
    imgs = sorted([f for f in os.listdir(person_dir) if f.endswith(".jpg")])
    label_map[idx] = person_name
    total = 0
    for fname in imgs:
        img = cv2.imread(os.path.join(person_dir, fname), cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        img     = cv2.resize(img, (200, 200))
        samples = augment(img, n=15)
        faces_train.extend(samples)
        labels_train.extend([idx] * len(samples))
        total += len(samples)
    print(f"  [OK] '{person_name}' → id={idx}  raw={len(imgs)}  total={total}")

print(f"\n[INFO] Total samples : {len(faces_train)}")
print(f"[INFO] Persons       : {list(label_map.values())}")

recognizer = cv2.face.LBPHFaceRecognizer_create(
    radius=2, neighbors=16, grid_x=8, grid_y=8
)
recognizer.train(faces_train, np.array(labels_train))
print("[INFO] Training complete.\n")

THRESHOLD = 85

def process_frame(frame):
    gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray  = cv2.equalizeHist(gray)
    faces = detector.detectMultiScale(
        gray, scaleFactor=1.05, minNeighbors=3,
        minSize=(50, 50), flags=cv2.CASCADE_SCALE_IMAGE
    )
    for (x, y, w, h) in faces:
        face_crop       = cv2.resize(gray[y:y+h, x:x+w], (200, 200))
        label_id, score = recognizer.predict(face_crop)
        print(f"  predict → '{label_map.get(label_id, '?')}'  score={score:.1f}")

        if score < THRESHOLD and label_id in label_map:
            name  = label_map[label_id]
            conf  = round(max(0, 100 - score), 1)
            color = (0, 200, 0)
        else:
            name  = "Unknown"
            conf  = 0.0
            color = (0, 0, 220)

        label = f"{name}  {conf}%" if conf else name
        cv2.rectangle(frame, (x, y),   (x+w, y+h),    color, 2)
        cv2.rectangle(frame, (x, y+h), (x+w, y+h+30), color, cv2.FILLED)
        cv2.putText(frame, label, (x+4, y+h+22),
                    cv2.FONT_HERSHEY_DUPLEX, 0.6, (255, 255, 255), 1)
    return frame

cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
if not cap.isOpened():
    cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("[ERROR] Cannot open camera.")
else:
    cap.set(cv2.CAP_PROP_FRAME_WIDTH,  1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    print("[INFO] Camera open. Press Q to quit.\n")
    for _ in range(15):
        cap.read()
    while True:
        ret, frame = cap.read()
        if not ret:
            time.sleep(0.03)
            continue
        cv2.imshow("Face Recognition  |  Q to quit", process_frame(frame))
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
    cap.release()
    cv2.destroyAllWindows()
    print("[INFO] Done.")

[INFO] Trying: c:\Users\AhmedJaber\AppData\Local\Programs\Python\Python311\Lib\site-packages\cv2\data\haarcascade_frontalface_default.xml
[INFO] Exists: False
[INFO] Downloading cascade file...
[INFO] Downloaded to d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition\haarcascade_frontalface_default.xml
[OK] Cascade loaded successfully from:
     d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition\haarcascade_frontalface_default.xml

[CAPTURING] 'Ahmad Imad'
  Press SPACE to capture (20 needed) | Q to finish early

  [SAVED] 1/20
  [SAVED] 2/20
  [SAVED] 3/20
  [SAVED] 4/20
  [SAVED] 5/20
  [SAVED] 6/20
  [SAVED] 7/20
  [SAVED] 8/20
  [SAVED] 9/20
  [SAVED] 10/20
  [SAVED] 11/20
  [SAVED] 12/20
  [SAVED] 13/20
  [SAVED] 14/20
  [SAVED] 15/20
  [SAVED] 16/20
  [SAVED] 17/20
  [SAVED] 18/20
  [SAVED] 19/20
  [SAVED] 20/20
  [DONE] 'Ahmad Imad' — 20 samples saved.


[CAPTURING] 'Ahmed Jaber'
  Press SPACE to capture (20 needed) | Q to finish early

  [SAVED] 1/20
  [SAVED] 2/20
  [SAVED] 3/

AttributeError: module 'cv2.face' has no attribute 'LBPHFaceRecognizer_create'

In [3]:
%pip uninstall opencv-python opencv-contrib-python opencv-python-headless -y
%pip install opencv-contrib-python==4.9.0.80

Found existing installation: opencv-contrib-python 4.13.0.92
Uninstalling opencv-contrib-python-4.13.0.92:
  Successfully uninstalled opencv-contrib-python-4.13.0.92
Note: you may need to restart the kernel to use updated packages.


   ---------------------------------------- 0.0/45.3 MB ? eta -:--:--
   ---------------------------------------- 0.3/45.3 MB ? eta -:--:--
   ------ --------------------------------- 7.9/45.3 MB 32.5 MB/s eta 0:00:02
   ---------- ----------------------------- 12.3/45.3 MB 27.6 MB/s eta 0:00:02
   --------------- ------------------------ 17.0/45.3 MB 26.2 MB/s eta 0:00:02
   --------------------- ------------------ 24.9/45.3 MB 30.9 MB/s eta 0:00:01
   ------------------------- -------------- 28.6/45.3 MB 26.7 MB/s eta 0:00:01
   ---------------------------- ----------- 32.5/45.3 MB 27.5 MB/s eta 0:00:01
   ---------------------------- ----------- 32.5/45.3 MB 27.5 MB/s eta 0:00:01
   ---------------------------- ----------- 32.5/45.3 MB 27.5 MB/s eta 0:00:01
   ---------------------------- ----------- 32.5/45.3 MB 27.5 MB/s eta 0:00:01
   ---------------------------- ----------- 32.5/45.3 MB 27.5 MB/s eta 0:00:01
   ---------------------------- ----------- 32.5/45.3 MB 27.5 MB/s eta 


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# ============================================================
#  CELL 0 — Run in TERMINAL (not Jupyter) — clean reinstall
#  Open cmd or PowerShell and paste these lines one by one:
# ============================================================
#
#  pip uninstall opencv-python opencv-contrib-python opencv-python-headless -y
#  pip install opencv-contrib-python==4.9.0.80
#
# Then restart VS Code / Jupyter completely and run Cell 1 below.


# ============================================================
#  CELL 1 — Verify install (run after terminal reinstall + restart)
# ============================================================
import cv2
print(f"[OK] cv2 version     : {cv2.__version__}")
print(f"[OK] CascadeClassifier exists : {hasattr(cv2, 'CascadeClassifier')}")
print(f"[OK] cv2.face exists          : {hasattr(cv2, 'face')}")


# ============================================================
#  CELL 2 — Train + Run (only run after Cell 1 shows all OK)
# ============================================================
import numpy as np
import cv2, os, time, urllib.request

PROJECT_DIR  = r"d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition"
SAMPLES_DIR  = os.path.join(PROJECT_DIR, "live_samples")
CASCADE_PATH = os.path.join(PROJECT_DIR, "haarcascade_frontalface_default.xml")

if not os.path.exists(CASCADE_PATH):
    print("[INFO] Downloading cascade...")
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml",
        CASCADE_PATH
    )

detector = cv2.CascadeClassifier(CASCADE_PATH)
assert not detector.empty(), "[ERROR] Cascade failed to load."
print(f"[OK] Cascade loaded from {CASCADE_PATH}")

PERSONS = ["Ahmad Imad", "Ahmed Jaber"]

def augment(face_gray, n=15):
    h, w    = face_gray.shape
    samples = [face_gray]
    for _ in range(n - 1):
        aug   = face_gray.copy()
        delta = np.random.randint(-45, 45)
        aug   = np.clip(aug.astype(np.int32) + delta, 0, 255).astype(np.uint8)
        if np.random.rand() > 0.5:
            aug = cv2.flip(aug, 1)
        angle = np.random.uniform(-20, 20)
        M     = cv2.getRotationMatrix2D((w//2, h//2), angle, 1.0)
        aug   = cv2.warpAffine(aug, M, (w, h))
        scale = np.random.uniform(0.80, 1.0)
        cw, ch = int(w*scale), int(h*scale)
        ox = np.random.randint(0, w - cw + 1)
        oy = np.random.randint(0, h - ch + 1)
        aug = cv2.resize(aug[oy:oy+ch, ox:ox+cw], (w, h))
        if np.random.rand() > 0.7:
            aug = cv2.GaussianBlur(aug, (3, 3), 0)
        samples.append(aug)
    return samples

faces_train  = []
labels_train = []
label_map    = {}

print("\n[INFO] Loading samples...\n")
for idx, person_name in enumerate(PERSONS):
    person_dir = os.path.join(SAMPLES_DIR, person_name)
    if not os.path.isdir(person_dir):
        print(f"  [SKIP] No folder for '{person_name}'")
        continue
    imgs = sorted([f for f in os.listdir(person_dir) if f.endswith(".jpg")])
    label_map[idx] = person_name
    total = 0
    for fname in imgs:
        img = cv2.imread(os.path.join(person_dir, fname), cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        img     = cv2.resize(img, (200, 200))
        samples = augment(img, n=15)
        faces_train.extend(samples)
        labels_train.extend([idx] * len(samples))
        total += len(samples)
    print(f"  [OK] '{person_name}' → id={idx}  raw={len(imgs)}  total={total}")

print(f"\n[INFO] Total : {len(faces_train)} samples | {list(label_map.values())}")

recognizer = cv2.face.LBPHFaceRecognizer_create(radius=2, neighbors=16, grid_x=8, grid_y=8)
recognizer.train(faces_train, np.array(labels_train))
print("[INFO] Training complete.\n")

THRESHOLD = 85

def process_frame(frame):
    gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray  = cv2.equalizeHist(gray)
    faces = detector.detectMultiScale(
        gray, scaleFactor=1.05, minNeighbors=3,
        minSize=(50, 50), flags=cv2.CASCADE_SCALE_IMAGE
    )
    for (x, y, w, h) in faces:
        face_crop       = cv2.resize(gray[y:y+h, x:x+w], (200, 200))
        label_id, score = recognizer.predict(face_crop)
        print(f"  predict → '{label_map.get(label_id,'?')}'  score={score:.1f}")
        if score < THRESHOLD and label_id in label_map:
            name  = label_map[label_id]
            conf  = round(max(0, 100 - score), 1)
            color = (0, 200, 0)
        else:
            name  = "Unknown"
            conf  = 0.0
            color = (0, 0, 220)
        label = f"{name}  {conf}%" if conf else name
        cv2.rectangle(frame, (x, y),   (x+w, y+h),    color, 2)
        cv2.rectangle(frame, (x, y+h), (x+w, y+h+30), color, cv2.FILLED)
        cv2.putText(frame, label, (x+4, y+h+22),
                    cv2.FONT_HERSHEY_DUPLEX, 0.6, (255, 255, 255), 1)
    return frame

cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
if not cap.isOpened():
    cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("[ERROR] Cannot open camera.")
else:
    cap.set(cv2.CAP_PROP_FRAME_WIDTH,  1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    print("[INFO] Camera open. Press Q to quit.\n")
    for _ in range(15):
        cap.read()
    while True:
        ret, frame = cap.read()
        if not ret:
            time.sleep(0.03)
            continue
        cv2.imshow("Face Recognition  |  Q to quit", process_frame(frame))
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
    cap.release()
    cv2.destroyAllWindows()
    print("[INFO] Done.")

AttributeError: module 'cv2' has no attribute '__version__'

In [1]:


# ============================================================
#  CELL 1 — Train + Run (after kernel restart)
# ============================================================
import numpy as np
import cv2, os, time

PROJECT_DIR  = r"d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition"
SAMPLES_DIR  = os.path.join(PROJECT_DIR, "live_samples")
CASCADE_PATH = os.path.join(PROJECT_DIR, "haarcascade_frontalface_default.xml")

# fallback download if missing
if not os.path.exists(CASCADE_PATH):
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml",
        CASCADE_PATH
    )

detector = cv2.CascadeClassifier(CASCADE_PATH)
assert not detector.empty(), "[ERROR] Cascade failed to load."
print(f"[OK] Cascade loaded.")
print(f"[OK] cv2.face available: {hasattr(cv2, 'face')}")

PERSONS = ["Ahmad Imad", "Ahmed Jaber"]

# ── Augmentation ─────────────────────────────────────────────
def augment(face_gray, n=15):
    h, w    = face_gray.shape
    samples = [face_gray]
    for _ in range(n - 1):
        aug   = face_gray.copy()
        delta = np.random.randint(-45, 45)
        aug   = np.clip(aug.astype(np.int32) + delta, 0, 255).astype(np.uint8)
        if np.random.rand() > 0.5:
            aug = cv2.flip(aug, 1)
        angle = np.random.uniform(-20, 20)
        M     = cv2.getRotationMatrix2D((w//2, h//2), angle, 1.0)
        aug   = cv2.warpAffine(aug, M, (w, h))
        scale = np.random.uniform(0.80, 1.0)
        cw, ch = int(w*scale), int(h*scale)
        ox = np.random.randint(0, w - cw + 1)
        oy = np.random.randint(0, h - ch + 1)
        aug = cv2.resize(aug[oy:oy+ch, ox:ox+cw], (w, h))
        if np.random.rand() > 0.7:
            aug = cv2.GaussianBlur(aug, (3, 3), 0)
        samples.append(aug)
    return samples

# ── Load samples ──────────────────────────────────────────────
faces_train  = []
labels_train = []
label_map    = {}

print("\n[INFO] Loading samples...\n")
for idx, person_name in enumerate(PERSONS):
    person_dir = os.path.join(SAMPLES_DIR, person_name)
    if not os.path.isdir(person_dir):
        print(f"  [SKIP] No folder for '{person_name}'")
        continue
    imgs  = sorted([f for f in os.listdir(person_dir) if f.endswith(".jpg")])
    label_map[idx] = person_name
    total = 0
    for fname in imgs:
        img = cv2.imread(os.path.join(person_dir, fname), cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        img     = cv2.resize(img, (200, 200))
        samples = augment(img, n=15)
        faces_train.extend(samples)
        labels_train.extend([idx] * len(samples))
        total += len(samples)
    print(f"  [OK] '{person_name}' → id={idx}  raw={len(imgs)}  total={total}")

print(f"\n[INFO] Total samples : {len(faces_train)}")
print(f"[INFO] Persons       : {list(label_map.values())}")

# ── Train ─────────────────────────────────────────────────────
recognizer = cv2.face.LBPHFaceRecognizer_create(
    radius=2, neighbors=16, grid_x=8, grid_y=8
)
recognizer.train(faces_train, np.array(labels_train))
print("[INFO] Training complete.\n")

# ── Live camera ───────────────────────────────────────────────
THRESHOLD = 85

def process_frame(frame):
    gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray  = cv2.equalizeHist(gray)
    faces = detector.detectMultiScale(
        gray, scaleFactor=1.05, minNeighbors=3,
        minSize=(50, 50), flags=cv2.CASCADE_SCALE_IMAGE
    )
    for (x, y, w, h) in faces:
        face_crop       = cv2.resize(gray[y:y+h, x:x+w], (200, 200))
        label_id, score = recognizer.predict(face_crop)
        print(f"  predict → '{label_map.get(label_id,'?')}'  score={score:.1f}")

        if score < THRESHOLD and label_id in label_map:
            name  = label_map[label_id]
            conf  = round(max(0, 100 - score), 1)
            color = (0, 200, 0)
        else:
            name  = "Unknown"
            conf  = 0.0
            color = (0, 0, 220)

        label = f"{name}  {conf}%" if conf else name
        cv2.rectangle(frame, (x, y),   (x+w, y+h),    color, 2)
        cv2.rectangle(frame, (x, y+h), (x+w, y+h+30), color, cv2.FILLED)
        cv2.putText(frame, label, (x+4, y+h+22),
                    cv2.FONT_HERSHEY_DUPLEX, 0.6, (255, 255, 255), 1)
    return frame

cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
if not cap.isOpened():
    cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("[ERROR] Cannot open camera.")
else:
    cap.set(cv2.CAP_PROP_FRAME_WIDTH,  1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    print("[INFO] Camera open. Press Q to quit.\n")
    for _ in range(15):
        cap.read()
    while True:
        ret, frame = cap.read()
        if not ret:
            time.sleep(0.03)
            continue
        cv2.imshow("Face Recognition  |  Q to quit", process_frame(frame))
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
    cap.release()
    cv2.destroyAllWindows()
    print("[INFO] Done.")

AttributeError: module 'cv2' has no attribute 'CascadeClassifier'

In [2]:
%pip uninstall opencv-python opencv-contrib-python opencv-python-headless facenet-pytorch deepface retina-face -y
%pip install opencv-contrib-python==4.9.0.80 numpy==2.4.4

Found existing installation: opencv-contrib-python 4.9.0.80
Uninstalling opencv-contrib-python-4.9.0.80:
  Successfully uninstalled opencv-contrib-python-4.9.0.80
Found existing installation: facenet-pytorch 2.6.0
Uninstalling facenet-pytorch-2.6.0:
  Successfully uninstalled facenet-pytorch-2.6.0
Found existing installation: deepface 0.0.99
Uninstalling deepface-0.0.99:
  Successfully uninstalled deepface-0.0.99
Found existing installation: retina-face 0.0.17
Uninstalling retina-face-0.0.17:
  Successfully uninstalled retina-face-0.0.17
Note: you may need to restart the kernel to use updated packages.


  Using cached opencv_contrib_python-4.9.0.80-cp37-abi3-win_amd64.whl.metadata (20 kB)
Using cached opencv_contrib_python-4.9.0.80-cp37-abi3-win_amd64.whl (45.3 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# ============================================================
#  CELL 0 — Verify ONLY (run first after terminal reinstall)
# ============================================================
import cv2
print(f"cv2 version  : {cv2.__version__}")
print(f"CascadeClassifier : {hasattr(cv2, 'CascadeClassifier')}")
print(f"cv2.face          : {hasattr(cv2, 'face')}")
print(f"LBPHFaceRecognizer: {hasattr(cv2.face, 'LBPHFaceRecognizer_create')}")

# If all 3 are True — run Cell 1 below
# If any is False  — run the terminal commands above again and restart VS Code


# ============================================================
#  CELL 1 — Train + Run (only after Cell 0 shows all True)
# ============================================================
import numpy as np
import cv2, os, time, urllib.request

PROJECT_DIR  = r"d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition"
SAMPLES_DIR  = os.path.join(PROJECT_DIR, "live_samples")
CASCADE_PATH = os.path.join(PROJECT_DIR, "haarcascade_frontalface_default.xml")

# ── Download cascade if missing ───────────────────────────────
if not os.path.exists(CASCADE_PATH):
    for url in [
        "https://raw.githubusercontent.com/opencv/opencv/4.x/data/haarcascades/haarcascade_frontalface_default.xml",
        "https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml",
    ]:
        try:
            print(f"[INFO] Downloading cascade from {url}")
            urllib.request.urlretrieve(url, CASCADE_PATH)
            print("[INFO] Download OK")
            break
        except Exception as e:
            print(f"[WARN] Failed: {e}")

detector = cv2.CascadeClassifier(CASCADE_PATH)
assert not detector.empty(), "[ERROR] Cascade failed — check the XML file exists"
print(f"[OK] Cascade loaded.")

# ── Only these two people are recognized — everyone else = Unknown ──
PERSONS = ["Ahmad Imad", "Ahmed Jaber"]

# ── Augmentation ──────────────────────────────────────────────
def augment(face_gray, n=15):
    h, w    = face_gray.shape
    samples = [face_gray]
    for _ in range(n - 1):
        aug   = face_gray.copy()
        delta = np.random.randint(-45, 45)
        aug   = np.clip(aug.astype(np.int32) + delta, 0, 255).astype(np.uint8)
        if np.random.rand() > 0.5:
            aug = cv2.flip(aug, 1)
        angle = np.random.uniform(-20, 20)
        M     = cv2.getRotationMatrix2D((w//2, h//2), angle, 1.0)
        aug   = cv2.warpAffine(aug, M, (w, h))
        scale = np.random.uniform(0.80, 1.0)
        cw, ch = int(w*scale), int(h*scale)
        ox = np.random.randint(0, w - cw + 1)
        oy = np.random.randint(0, h - ch + 1)
        aug = cv2.resize(aug[oy:oy+ch, ox:ox+cw], (w, h))
        if np.random.rand() > 0.7:
            aug = cv2.GaussianBlur(aug, (3, 3), 0)
        samples.append(aug)
    return samples

# ── Load live samples ─────────────────────────────────────────
faces_train  = []
labels_train = []
label_map    = {}

print("\n[INFO] Loading samples...\n")
for idx, person_name in enumerate(PERSONS):
    person_dir = os.path.join(SAMPLES_DIR, person_name)
    if not os.path.isdir(person_dir):
        print(f"  [SKIP] No folder for '{person_name}'")
        continue
    imgs = sorted([f for f in os.listdir(person_dir) if f.endswith(".jpg")])
    if not imgs:
        print(f"  [SKIP] No samples for '{person_name}'")
        continue
    label_map[idx] = person_name
    total = 0
    for fname in imgs:
        img = cv2.imread(os.path.join(person_dir, fname), cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        img     = cv2.resize(img, (200, 200))
        samples = augment(img, n=15)
        faces_train.extend(samples)
        labels_train.extend([idx] * len(samples))
        total += len(samples)
    print(f"  [OK] '{person_name}' → id={idx}  raw={len(imgs)}  augmented={total}")

print(f"\n[INFO] Total : {len(faces_train)} samples")
print(f"[INFO] People: {list(label_map.values())}")

# ── Train ──────────────────────────────────────────────────────
recognizer = cv2.face.LBPHFaceRecognizer_create(
    radius=2, neighbors=16, grid_x=8, grid_y=8
)
recognizer.train(faces_train, np.array(labels_train))
print("[INFO] Training complete.\n")

# ── Live recognition ───────────────────────────────────────────
THRESHOLD = 85   # watch printed scores — adjust if needed

def process_frame(frame):
    gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray  = cv2.equalizeHist(gray)
    faces = detector.detectMultiScale(
        gray, scaleFactor=1.05, minNeighbors=3,
        minSize=(50, 50), flags=cv2.CASCADE_SCALE_IMAGE
    )
    for (x, y, w, h) in faces:
        face_crop       = cv2.resize(gray[y:y+h, x:x+w], (200, 200))
        label_id, score = recognizer.predict(face_crop)
        print(f"  predict → '{label_map.get(label_id,'?')}'  score={score:.1f}")

        if score < THRESHOLD and label_id in label_map:
            name  = label_map[label_id]
            conf  = round(max(0, 100 - score), 1)
            color = (0, 200, 0)     # green
        else:
            name  = "Unknown"
            conf  = 0.0
            color = (0, 0, 220)     # red

        label = f"{name}  {conf}%" if conf else name
        cv2.rectangle(frame, (x, y),   (x+w, y+h),    color, 2)
        cv2.rectangle(frame, (x, y+h), (x+w, y+h+30), color, cv2.FILLED)
        cv2.putText(frame, label, (x+4, y+h+22),
                    cv2.FONT_HERSHEY_DUPLEX, 0.6, (255, 255, 255), 1)
    return frame

cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
if not cap.isOpened():
    cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("[ERROR] Cannot open camera.")
else:
    cap.set(cv2.CAP_PROP_FRAME_WIDTH,  1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    print("[INFO] Camera open. Press Q to quit.\n")
    for _ in range(15):
        cap.read()
    while True:
        ret, frame = cap.read()
        if not ret:
            time.sleep(0.03)
            continue
        cv2.imshow("Face Recognition  |  Q to quit", process_frame(frame))
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
    cap.release()
    cv2.destroyAllWindows()
    print("[INFO] Done.")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\AhmedJaber\AppData\Roaming\Python\Python311\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\AhmedJaber\AppData\Roaming\Python\Python311\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\AhmedJaber\AppData\Roaming\Python\Python311\site-packages\ipykernel\kernelapp.py", line 739, in start

AttributeError: _ARRAY_API not found

ImportError: numpy.core.multiarray failed to import

In [5]:
%pip uninstall opencv-python opencv-contrib-python opencv-python-headless facenet-pytorch deepface retina-face numpy -y
%pip install numpy==2.1.0
%pip install opencv-contrib-python==4.9.0.80

Found existing installation: opencv-contrib-python 4.9.0.80
Uninstalling opencv-contrib-python-4.9.0.80:
  Successfully uninstalled opencv-contrib-python-4.9.0.80
Found existing installation: numpy 2.4.4
Uninstalling numpy-2.4.4:
  Successfully uninstalled numpy-2.4.4
Note: you may need to restart the kernel to use updated packages.


You can safely remove it manually.
You can safely remove it manually.
You can safely remove it manually.


   ---------------------------------------- 0.0/12.9 MB ? eta -:--:--
   -- ------------------------------------- 0.8/12.9 MB 6.7 MB/s eta 0:00:02
   --------- ------------------------------ 3.1/12.9 MB 12.3 MB/s eta 0:00:01
   --------- ------------------------------ 3.1/12.9 MB 12.3 MB/s eta 0:00:01
   --------- ------------------------------ 3.1/12.9 MB 12.3 MB/s eta 0:00:01
   --------- ------------------------------ 3.1/12.9 MB 12.3 MB/s eta 0:00:01
   --------- ------------------------------ 3.1/12.9 MB 12.3 MB/s eta 0:00:01
   --------- ------------------------------ 3.1/12.9 MB 12.3 MB/s eta 0:00:01
   --------- ------------------------------ 3.1/12.9 MB 12.3 MB/s eta 0:00:01
   --------- ------------------------------ 3.1/12.9 MB 12.3 MB/s eta 0:00:01
   --------- ------------------------------ 3.1/12.9 MB 12.3 MB/s eta 0:00:01
   --------- ------------------------------ 3.1/12.9 MB 12.3 MB/s eta 0:00:01
   --------- ------------------------------ 3.1/12.9 MB 12.3 MB/s eta 0:0

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mediapipe 0.10.35 requires opencv-contrib-python, which is not installed.

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached opencv_contrib_python-4.9.0.80-cp37-abi3-win_amd64.whl.metadata (20 kB)
Using cached opencv_contrib_python-4.9.0.80-cp37-abi3-win_amd64.whl (45.3 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# ============================================================
#  CELL 0 — Verify ONLY (run first after terminal reinstall)
# ============================================================
import cv2
print(f"cv2 version  : {cv2.__version__}")
print(f"CascadeClassifier : {hasattr(cv2, 'CascadeClassifier')}")
print(f"cv2.face          : {hasattr(cv2, 'face')}")
print(f"LBPHFaceRecognizer: {hasattr(cv2.face, 'LBPHFaceRecognizer_create')}")

# If all 3 are True — run Cell 1 below
# If any is False  — run the terminal commands above again and restart VS Code


# ============================================================
#  CELL 1 — Train + Run (only after Cell 0 shows all True)
# ============================================================
import numpy as np
import cv2, os, time, urllib.request

PROJECT_DIR  = r"d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition"
SAMPLES_DIR  = os.path.join(PROJECT_DIR, "live_samples")
CASCADE_PATH = os.path.join(PROJECT_DIR, "haarcascade_frontalface_default.xml")

# ── Download cascade if missing ───────────────────────────────
if not os.path.exists(CASCADE_PATH):
    for url in [
        "https://raw.githubusercontent.com/opencv/opencv/4.x/data/haarcascades/haarcascade_frontalface_default.xml",
        "https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml",
    ]:
        try:
            print(f"[INFO] Downloading cascade from {url}")
            urllib.request.urlretrieve(url, CASCADE_PATH)
            print("[INFO] Download OK")
            break
        except Exception as e:
            print(f"[WARN] Failed: {e}")

detector = cv2.CascadeClassifier(CASCADE_PATH)
assert not detector.empty(), "[ERROR] Cascade failed — check the XML file exists"
print(f"[OK] Cascade loaded.")

# ── Only these two people are recognized — everyone else = Unknown ──
PERSONS = ["Ahmad Imad", "Ahmed Jaber"]

# ── Augmentation ──────────────────────────────────────────────
def augment(face_gray, n=15):
    h, w    = face_gray.shape
    samples = [face_gray]
    for _ in range(n - 1):
        aug   = face_gray.copy()
        delta = np.random.randint(-45, 45)
        aug   = np.clip(aug.astype(np.int32) + delta, 0, 255).astype(np.uint8)
        if np.random.rand() > 0.5:
            aug = cv2.flip(aug, 1)
        angle = np.random.uniform(-20, 20)
        M     = cv2.getRotationMatrix2D((w//2, h//2), angle, 1.0)
        aug   = cv2.warpAffine(aug, M, (w, h))
        scale = np.random.uniform(0.80, 1.0)
        cw, ch = int(w*scale), int(h*scale)
        ox = np.random.randint(0, w - cw + 1)
        oy = np.random.randint(0, h - ch + 1)
        aug = cv2.resize(aug[oy:oy+ch, ox:ox+cw], (w, h))
        if np.random.rand() > 0.7:
            aug = cv2.GaussianBlur(aug, (3, 3), 0)
        samples.append(aug)
    return samples

# ── Load live samples ─────────────────────────────────────────
faces_train  = []
labels_train = []
label_map    = {}

print("\n[INFO] Loading samples...\n")
for idx, person_name in enumerate(PERSONS):
    person_dir = os.path.join(SAMPLES_DIR, person_name)
    if not os.path.isdir(person_dir):
        print(f"  [SKIP] No folder for '{person_name}'")
        continue
    imgs = sorted([f for f in os.listdir(person_dir) if f.endswith(".jpg")])
    if not imgs:
        print(f"  [SKIP] No samples for '{person_name}'")
        continue
    label_map[idx] = person_name
    total = 0
    for fname in imgs:
        img = cv2.imread(os.path.join(person_dir, fname), cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        img     = cv2.resize(img, (200, 200))
        samples = augment(img, n=15)
        faces_train.extend(samples)
        labels_train.extend([idx] * len(samples))
        total += len(samples)
    print(f"  [OK] '{person_name}' → id={idx}  raw={len(imgs)}  augmented={total}")

print(f"\n[INFO] Total : {len(faces_train)} samples")
print(f"[INFO] People: {list(label_map.values())}")

# ── Train ──────────────────────────────────────────────────────
recognizer = cv2.face.LBPHFaceRecognizer_create(
    radius=2, neighbors=16, grid_x=8, grid_y=8
)
recognizer.train(faces_train, np.array(labels_train))
print("[INFO] Training complete.\n")

# ── Live recognition ───────────────────────────────────────────
THRESHOLD = 85   # watch printed scores — adjust if needed

def process_frame(frame):
    gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray  = cv2.equalizeHist(gray)
    faces = detector.detectMultiScale(
        gray, scaleFactor=1.05, minNeighbors=3,
        minSize=(50, 50), flags=cv2.CASCADE_SCALE_IMAGE
    )
    for (x, y, w, h) in faces:
        face_crop       = cv2.resize(gray[y:y+h, x:x+w], (200, 200))
        label_id, score = recognizer.predict(face_crop)
        print(f"  predict → '{label_map.get(label_id,'?')}'  score={score:.1f}")

        if score < THRESHOLD and label_id in label_map:
            name  = label_map[label_id]
            conf  = round(max(0, 100 - score), 1)
            color = (0, 200, 0)     # green
        else:
            name  = "Unknown"
            conf  = 0.0
            color = (0, 0, 220)     # red

        label = f"{name}  {conf}%" if conf else name
        cv2.rectangle(frame, (x, y),   (x+w, y+h),    color, 2)
        cv2.rectangle(frame, (x, y+h), (x+w, y+h+30), color, cv2.FILLED)
        cv2.putText(frame, label, (x+4, y+h+22),
                    cv2.FONT_HERSHEY_DUPLEX, 0.6, (255, 255, 255), 1)
    return frame

cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
if not cap.isOpened():
    cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("[ERROR] Cannot open camera.")
else:
    cap.set(cv2.CAP_PROP_FRAME_WIDTH,  1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    print("[INFO] Camera open. Press Q to quit.\n")
    for _ in range(15):
        cap.read()
    while True:
        ret, frame = cap.read()
        if not ret:
            time.sleep(0.03)
            continue
        cv2.imshow("Face Recognition  |  Q to quit", process_frame(frame))
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
    cap.release()
    cv2.destroyAllWindows()
    print("[INFO] Done.")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.0 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\AhmedJaber\AppData\Roaming\Python\Python311\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\AhmedJaber\AppData\Roaming\Python\Python311\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\AhmedJaber\AppData\Roaming\Python\Python311\site-packages\ipykernel\kernelapp.py", line 739, in start

AttributeError: _ARRAY_API not found

ImportError: numpy.core.multiarray failed to import

In [2]:
%pip uninstall numpy opencv-python opencv-contrib-python opencv-python-headless facenet-pytorch deepface retina-face mtcnn tf-keras -y
%pip install numpy==2.1.0
%pip install opencv-contrib-python==4.9.0.80

Found existing installation: numpy 2.1.0
Uninstalling numpy-2.1.0:
  Successfully uninstalled numpy-2.1.0
Found existing installation: opencv-contrib-python 4.9.0.80
Uninstalling opencv-contrib-python-4.9.0.80:
  Successfully uninstalled opencv-contrib-python-4.9.0.80
Found existing installation: mtcnn 1.0.0
Uninstalling mtcnn-1.0.0:
  Successfully uninstalled mtcnn-1.0.0
Found existing installation: tf_keras 2.21.0
Uninstalling tf_keras-2.21.0:
  Successfully uninstalled tf_keras-2.21.0
Note: you may need to restart the kernel to use updated packages.


You can safely remove it manually.
You can safely remove it manually.
You can safely remove it manually.


  Using cached numpy-2.1.0-cp311-cp311-win_amd64.whl.metadata (59 kB)
Using cached numpy-2.1.0-cp311-cp311-win_amd64.whl (12.9 MB)
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mediapipe 0.10.35 requires opencv-contrib-python, which is not installed.

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached opencv_contrib_python-4.9.0.80-cp37-abi3-win_amd64.whl.metadata (20 kB)
Using cached opencv_contrib_python-4.9.0.80-cp37-abi3-win_amd64.whl (45.3 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
pip install opencv-contrib-python

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
# ============================================================
#  CELL 1 — Capture 20 live webcam samples per person
# ============================================================
import os, time
import numpy as np
import cv2

PROJECT_DIR  = r"d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition"
SAMPLES_DIR  = os.path.join(PROJECT_DIR, "live_samples")
CASCADE_PATH = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
os.makedirs(SAMPLES_DIR, exist_ok=True)

detector = cv2.CascadeClassifier(CASCADE_PATH)

# ── Edit this list — one entry per person to enroll ──────────
PERSONS = ["Ahmad Imad", "Ahmed Ashraf", "Ahmed Jaber", "Ahmed Kamel"]
SAMPLES_PER_PERSON = 20

def capture_person(name, n):
    save_dir = os.path.join(SAMPLES_DIR, name)
    os.makedirs(save_dir, exist_ok=True)

    cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
    if not cap.isOpened():
        cap = cv2.VideoCapture(0)

    cap.set(cv2.CAP_PROP_FRAME_WIDTH,  1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

    # warm-up
    for _ in range(15):
        cap.read()

    count   = 0
    print(f"\n[CAPTURING] '{name}' — look at the camera.")
    print(f"  Press SPACE to capture a sample ({n} needed) | Q to skip this person\n")

    while count < n:
        ret, frame = cap.read()
        if not ret:
            continue

        gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray  = cv2.equalizeHist(gray)
        faces = detector.detectMultiScale(
            gray, scaleFactor=1.05, minNeighbors=3, minSize=(50, 50)
        )

        display = frame.copy()
        face_found = False

        for (x, y, w, h) in faces:
            face_found = True
            cv2.rectangle(display, (x, y), (x+w, y+h), (0, 255, 0), 2)

        status = f"'{name}'  captured: {count}/{n}  |  SPACE=capture  Q=skip"
        hint   = "Face detected — ready!" if face_found else "No face detected — move closer"
        color  = (0, 200, 0) if face_found else (0, 0, 220)

        cv2.putText(display, status, (10, 30),
                    cv2.FONT_HERSHEY_DUPLEX, 0.7, (255, 255, 255), 1)
        cv2.putText(display, hint, (10, 65),
                    cv2.FONT_HERSHEY_DUPLEX, 0.65, color, 1)

        cv2.imshow(f"Enrolling: {name}  |  SPACE=capture  Q=skip", display)
        key = cv2.waitKey(1) & 0xFF

        if key == ord(" ") and face_found:
            # save the face crop
            x, y, w, h = faces[0]
            face_crop  = cv2.resize(gray[y:y+h, x:x+w], (200, 200))
            fname      = os.path.join(save_dir, f"{count:03d}.jpg")
            cv2.imwrite(fname, face_crop)
            count += 1
            print(f"  [SAVED] {fname}")
            time.sleep(0.3)   # small pause between captures

        elif key == ord("q"):
            print(f"  [SKIPPED] '{name}'")
            break

    cap.release()
    cv2.destroyAllWindows()
    print(f"  [DONE] '{name}' — {count} samples saved to {save_dir}")
    return count

# ── Run capture for each person ───────────────────────────────
for person in PERSONS:
    existing = [f for f in os.listdir(os.path.join(SAMPLES_DIR, person))
                if f.endswith(".jpg")] if os.path.isdir(
                    os.path.join(SAMPLES_DIR, person)) else []
    if len(existing) >= SAMPLES_PER_PERSON:
        print(f"[SKIP] '{person}' already has {len(existing)} samples.")
        continue
    capture_person(person, SAMPLES_PER_PERSON)

print("\n[ALL DONE] Run Cell 2 to train and start recognition.")


# ============================================================
#  CELL 2 — Train on live samples + run camera
# ============================================================
import json
import numpy as np
import cv2
import os
import time

PROJECT_DIR  = r"d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition"
SAMPLES_DIR  = os.path.join(PROJECT_DIR, "live_samples")
CASCADE_PATH = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
detector     = cv2.CascadeClassifier(CASCADE_PATH)

# ── Load samples ──────────────────────────────────────────────
faces_train  = []
labels_train = []
label_map    = {}
label_id     = 0

for person_name in sorted(os.listdir(SAMPLES_DIR)):
    person_dir = os.path.join(SAMPLES_DIR, person_name)
    if not os.path.isdir(person_dir):
        continue

    imgs = [f for f in os.listdir(person_dir) if f.endswith(".jpg")]
    if len(imgs) == 0:
        continue

    label_map[label_id] = person_name
    count = 0
    for fname in imgs:
        img = cv2.imread(os.path.join(person_dir, fname), cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        img = cv2.resize(img, (200, 200))
        faces_train.append(img)
        labels_train.append(label_id)
        count += 1

    print(f"  [OK] '{person_name}' → id={label_id}  samples={count}")
    label_id += 1

print(f"\n[INFO] Total samples: {len(faces_train)} | Persons: {len(label_map)}")

# ── Train ─────────────────────────────────────────────────────
recognizer = cv2.face.LBPHFaceRecognizer_create(
    radius=2, neighbors=16, grid_x=8, grid_y=8
)
recognizer.train(faces_train, np.array(labels_train))
print(f"[INFO] Training complete. Enrolled: {list(label_map.values())}")

# ── Live recognition ──────────────────────────────────────────
# Start with 100 — watch printed scores and adjust
THRESHOLD = 100

def process_frame(frame):
    gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray  = cv2.equalizeHist(gray)
    faces = detector.detectMultiScale(
        gray, scaleFactor=1.05, minNeighbors=3,
        minSize=(50, 50), flags=cv2.CASCADE_SCALE_IMAGE
    )
    for (x, y, w, h) in faces:
        face_crop       = cv2.resize(gray[y:y+h, x:x+w], (200, 200))
        label_id, score = recognizer.predict(face_crop)
        print(f"  predict → '{label_map.get(label_id,'?')}'  score={score:.1f}")

        if score < THRESHOLD:
            name  = label_map.get(label_id, "Unknown")
            conf  = round(max(0, 100 - score), 1)
            color = (0, 200, 0)
        else:
            name  = "Unknown"
            conf  = 0.0
            color = (0, 0, 220)

        label = f"{name}  {conf}%" if conf else name
        cv2.rectangle(frame, (x, y),   (x+w, y+h),    color, 2)
        cv2.rectangle(frame, (x, y+h), (x+w, y+h+30), color, cv2.FILLED)
        cv2.putText(frame, label, (x+4, y+h+22),
                    cv2.FONT_HERSHEY_DUPLEX, 0.6, (255, 255, 255), 1)
    return frame

cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
if not cap.isOpened():
    cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("[ERROR] Cannot open camera.")
else:
    cap.set(cv2.CAP_PROP_FRAME_WIDTH,  1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    print("\n[INFO] Camera open. Press Q to quit.\n")
    for _ in range(15):
        cap.read()
    while True:
        ret, frame = cap.read()
        if not ret:
            time.sleep(0.03)
            continue
        cv2.imshow("Face Recognition  |  Q to quit", process_frame(frame))
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
    cap.release()
    cv2.destroyAllWindows()
    print("[INFO] Done.")

[SKIP] 'Ahmad Imad' already has 160 samples.
[SKIP] 'Ahmed Ashraf' already has 160 samples.
[SKIP] 'Ahmed Jaber' already has 160 samples.
[SKIP] 'Ahmed Kamel' already has 160 samples.

[ALL DONE] Run Cell 2 to train and start recognition.
  [OK] 'Ahmad Imad' → id=0  samples=160
  [OK] 'Ahmed Ashraf' → id=1  samples=160
  [OK] 'Ahmed Jaber' → id=2  samples=160
  [OK] 'Ahmed Kamel' → id=3  samples=160

[INFO] Total samples: 640 | Persons: 4


AttributeError: module 'cv2.face' has no attribute 'LBPHFaceRecognizer_create'

In [1]:
# ============================================================
#  CELL 0 — Verify ONLY (run first after terminal reinstall)
# ============================================================
import cv2
print(f"cv2 version  : {cv2.__version__}")
print(f"CascadeClassifier : {hasattr(cv2, 'CascadeClassifier')}")
print(f"cv2.face          : {hasattr(cv2, 'face')}")
print(f"LBPHFaceRecognizer: {hasattr(cv2.face, 'LBPHFaceRecognizer_create')}")

# If all 3 are True — run Cell 1 below
# If any is False  — run the terminal commands above again and restart VS Code


# ============================================================
#  CELL 1 — Train + Run (only after Cell 0 shows all True)
# ============================================================
import numpy as np
import cv2, os, time, urllib.request

PROJECT_DIR  = r"d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition"
SAMPLES_DIR  = os.path.join(PROJECT_DIR, "live_samples")
CASCADE_PATH = os.path.join(PROJECT_DIR, "haarcascade_frontalface_default.xml")

# ── Download cascade if missing ───────────────────────────────
if not os.path.exists(CASCADE_PATH):
    for url in [
        "https://raw.githubusercontent.com/opencv/opencv/4.x/data/haarcascades/haarcascade_frontalface_default.xml",
        "https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml",
    ]:
        try:
            print(f"[INFO] Downloading cascade from {url}")
            urllib.request.urlretrieve(url, CASCADE_PATH)
            print("[INFO] Download OK")
            break
        except Exception as e:
            print(f"[WARN] Failed: {e}")

detector = cv2.CascadeClassifier(CASCADE_PATH)
assert not detector.empty(), "[ERROR] Cascade failed — check the XML file exists"
print(f"[OK] Cascade loaded.")

# ── Only these two people are recognized — everyone else = Unknown ──
PERSONS = ["Ahmad Imad", "Ahmed Jaber"]

# ── Augmentation ──────────────────────────────────────────────
def augment(face_gray, n=15):
    h, w    = face_gray.shape
    samples = [face_gray]
    for _ in range(n - 1):
        aug   = face_gray.copy()
        delta = np.random.randint(-45, 45)
        aug   = np.clip(aug.astype(np.int32) + delta, 0, 255).astype(np.uint8)
        if np.random.rand() > 0.5:
            aug = cv2.flip(aug, 1)
        angle = np.random.uniform(-20, 20)
        M     = cv2.getRotationMatrix2D((w//2, h//2), angle, 1.0)
        aug   = cv2.warpAffine(aug, M, (w, h))
        scale = np.random.uniform(0.80, 1.0)
        cw, ch = int(w*scale), int(h*scale)
        ox = np.random.randint(0, w - cw + 1)
        oy = np.random.randint(0, h - ch + 1)
        aug = cv2.resize(aug[oy:oy+ch, ox:ox+cw], (w, h))
        if np.random.rand() > 0.7:
            aug = cv2.GaussianBlur(aug, (3, 3), 0)
        samples.append(aug)
    return samples

# ── Load live samples ─────────────────────────────────────────
faces_train  = []
labels_train = []
label_map    = {}

print("\n[INFO] Loading samples...\n")
for idx, person_name in enumerate(PERSONS):
    person_dir = os.path.join(SAMPLES_DIR, person_name)
    if not os.path.isdir(person_dir):
        print(f"  [SKIP] No folder for '{person_name}'")
        continue
    imgs = sorted([f for f in os.listdir(person_dir) if f.endswith(".jpg")])
    if not imgs:
        print(f"  [SKIP] No samples for '{person_name}'")
        continue
    label_map[idx] = person_name
    total = 0
    for fname in imgs:
        img = cv2.imread(os.path.join(person_dir, fname), cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        img     = cv2.resize(img, (200, 200))
        samples = augment(img, n=15)
        faces_train.extend(samples)
        labels_train.extend([idx] * len(samples))
        total += len(samples)
    print(f"  [OK] '{person_name}' → id={idx}  raw={len(imgs)}  augmented={total}")

print(f"\n[INFO] Total : {len(faces_train)} samples")
print(f"[INFO] People: {list(label_map.values())}")

# ── Train ──────────────────────────────────────────────────────
recognizer = cv2.face.LBPHFaceRecognizer_create(
    radius=2, neighbors=16, grid_x=8, grid_y=8
)
recognizer.train(faces_train, np.array(labels_train))
print("[INFO] Training complete.\n")

# ── Live recognition ───────────────────────────────────────────
THRESHOLD = 85   # watch printed scores — adjust if needed

def process_frame(frame):
    gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray  = cv2.equalizeHist(gray)
    faces = detector.detectMultiScale(
        gray, scaleFactor=1.05, minNeighbors=3,
        minSize=(50, 50), flags=cv2.CASCADE_SCALE_IMAGE
    )
    for (x, y, w, h) in faces:
        face_crop       = cv2.resize(gray[y:y+h, x:x+w], (200, 200))
        label_id, score = recognizer.predict(face_crop)
        print(f"  predict → '{label_map.get(label_id,'?')}'  score={score:.1f}")

        if score < THRESHOLD and label_id in label_map:
            name  = label_map[label_id]
            conf  = round(max(0, 100 - score), 1)
            color = (0, 200, 0)     # green
        else:
            name  = "Unknown"
            conf  = 0.0
            color = (0, 0, 220)     # red

        label = f"{name}  {conf}%" if conf else name
        cv2.rectangle(frame, (x, y),   (x+w, y+h),    color, 2)
        cv2.rectangle(frame, (x, y+h), (x+w, y+h+30), color, cv2.FILLED)
        cv2.putText(frame, label, (x+4, y+h+22),
                    cv2.FONT_HERSHEY_DUPLEX, 0.6, (255, 255, 255), 1)
    return frame

cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
if not cap.isOpened():
    cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("[ERROR] Cannot open camera.")
else:
    cap.set(cv2.CAP_PROP_FRAME_WIDTH,  1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    print("[INFO] Camera open. Press Q to quit.\n")
    for _ in range(15):
        cap.read()
    while True:
        ret, frame = cap.read()
        if not ret:
            time.sleep(0.03)
            continue
        cv2.imshow("Face Recognition  |  Q to quit", process_frame(frame))
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
    cap.release()
    cv2.destroyAllWindows()
    print("[INFO] Done.")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.0 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\AhmedJaber\AppData\Roaming\Python\Python311\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\AhmedJaber\AppData\Roaming\Python\Python311\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\AhmedJaber\AppData\Roaming\Python\Python311\site-packages\ipykernel\kernelapp.py", line 739, in start

AttributeError: _ARRAY_API not found

ImportError: numpy.core.multiarray failed to import

In [5]:
python -m venv d:\cv_env
d:\cv_env\Scripts\activate
pip install numpy==2.1.0 opencv-contrib-python==4.9.0.80 pillow

SyntaxError: invalid syntax (848771737.py, line 1)

In [18]:




%pip uninstall opencv-python -y
%pip install opencv-contrib-python

Found existing installation: opencv-python 4.13.0.92
Uninstalling opencv-python-4.13.0.92:
  Successfully uninstalled opencv-python-4.13.0.92
Note: you may need to restart the kernel to use updated packages.


You can safely remove it manually.


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
import os
import time
import numpy as np
import cv2
import urllib.request

# --- 1. SETUP PATHS ---
PROJECT_DIR  = r"D:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition"
SAMPLES_DIR  = os.path.join(PROJECT_DIR, "live_samples")
os.makedirs(SAMPLES_DIR, exist_ok=True)

# The local path where we WANT the file to be
CASCADE_PATH = os.path.join(PROJECT_DIR, "haarcascade_frontalface_default.xml")

# --- 2. SELF-HEALING: AUTO-DOWNLOAD IF MISSING ---
if not os.path.exists(CASCADE_PATH):
    print(f"[INFO] XML file missing. Attempting auto-download to {PROJECT_DIR}...")
    # Using the official OpenCV 'main' branch URL
    url = "https://raw.githubusercontent.com/opencv/opencv/main/data/haarcascades/haarcascade_frontalface_default.xml"
    try:
        urllib.request.urlretrieve(url, CASCADE_PATH)
        print("[SUCCESS] XML file downloaded successfully!")
    except Exception as e:
        print(f"[ERROR] Auto-download failed: {e}")
        print("Please check your internet connection or download the file manually.")

# Initialize Detector
detector = cv2.CascadeClassifier(CASCADE_PATH)

if detector.empty():
    print(f"\n[CRITICAL] Still failed to load. Check if '{CASCADE_PATH}' is a valid file.")
else:
    print("[INFO] Face Detector is READY.")

# --- 3. HACKATHON TEAM ---
PERSONS = ["Ahmad Imad", "Ahmed Jaber"]
SAMPLES_PER_PERSON = 20

# --- 4. CAPTURE LOGIC ---
def capture_person(name, n):
    save_dir = os.path.join(SAMPLES_DIR, name)
    os.makedirs(save_dir, exist_ok=True)

    # Open camera with DSHOW for faster startup on Windows
    cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
    if not cap.isOpened():
        cap = cv2.VideoCapture(0) # Fallback

    # Warm-up camera
    for _ in range(10): cap.read()

    count = 0
    print(f"\n[CAPTURING] '{name}' — look at the camera.")
    print(f"  Press SPACE to capture a sample ({n} needed) | Q to skip\n")

    while count < n:
        ret, frame = cap.read()
        if not ret: continue

        gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray  = cv2.equalizeHist(gray)
        
        # Detection
        faces = detector.detectMultiScale(gray, 1.1, 5, minSize=(100, 100))

        display = frame.copy()
        face_found = False

        for (x, y, w, h) in faces:
            face_found = True
            cv2.rectangle(display, (x, y), (x+w, y+h), (0, 255, 0), 2)

        status = f"'{name}' {count}/{n} | SPACE=capture Q=skip"
        color  = (0, 255, 0) if face_found else (0, 0, 255)
        
        cv2.putText(display, status, (10, 30), cv2.FONT_HERSHEY_DUPLEX, 0.7, color, 2)
        cv2.imshow("Hackathon Enrollment", display)
        
        key = cv2.waitKey(1) & 0xFF

        if key == ord(" ") and face_found:
            x, y, w, h = faces[0]
            face_crop = cv2.resize(gray[y:y+h, x:x+w], (200, 200))
            fname = os.path.join(save_dir, f"{count:03d}.jpg")
            cv2.imwrite(fname, face_crop)
            count += 1
            print(f"  [SAVED] {count}/{n}")
            time.sleep(0.1)
        elif key == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()
    return count

# --- 5. RUN ---
if not detector.empty():
    for person in PERSONS:
        capture_person(person, SAMPLES_PER_PERSON)
    print("\n[ALL DONE] New 20 samples captured for each person!")

[INFO] XML file missing. Attempting auto-download to D:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition...
[ERROR] Auto-download failed: HTTP Error 404: Not Found
Please check your internet connection or download the file manually.

[CRITICAL] Still failed to load. Check if 'D:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition\haarcascade_frontalface_default.xml' is a valid file.


In [14]:
# ============================================================
#  FINAL CELL — Train and Run with "Unknown" Rejection
# ============================================================
import numpy as np
import cv2
import os
import time

PROJECT_DIR  = r"d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition"
SAMPLES_DIR  = os.path.join(PROJECT_DIR, "live_samples")
CASCADE_PATH = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
detector     = cv2.CascadeClassifier(CASCADE_PATH)

# ── 1. Load Augmented Dataset ───────────────────────────────
faces_train  = []
labels_train = []
label_map    = {}
label_id     = 0

print("[SYSTEM] Loading face database...")
for person_name in sorted(os.listdir(SAMPLES_DIR)):
    person_dir = os.path.join(SAMPLES_DIR, person_name)
    if not os.path.isdir(person_dir): continue

    imgs = [f for f in os.listdir(person_dir) if f.endswith(".jpg")]
    if not imgs: continue

    label_map[label_id] = person_name
    for fname in imgs:
        img = cv2.imread(os.path.join(person_dir, fname), cv2.IMREAD_GRAYSCALE)
        if img is not None:
            faces_train.append(cv2.resize(img, (200, 200)))
            labels_train.append(label_id)
    print(f"  [LOADED] {person_name} (ID: {label_id})")
    label_id += 1

# ── 2. Train the Recognizer ─────────────────────────────────
recognizer = cv2.face.LBPHFaceRecognizer_create(radius=1, neighbors=8, grid_x=8, grid_y=8)
recognizer.train(faces_train, np.array(labels_train))
print(f"\n[SYSTEM] Training complete on {len(faces_train)} samples.")

# ── 3. Calibration & Logic ──────────────────────────────────
# HACKATHON TIP: 
# - If you see your score is 70-90, but a stranger is 120+, set this to 105.
# - The lower this number, the "stricter" the security.
STRICT_THRESHOLD = 110 

def process_frame(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = cv2.equalizeHist(gray)
    faces = detector.detectMultiScale(gray, 1.1, 5, minSize=(100, 100))

    for (x, y, w, h) in faces:
        face_crop = cv2.resize(gray[y:y+h, x:x+w], (200, 200))
        label_id, score = recognizer.predict(face_crop)

        # Logic for identifying vs rejecting
        if score < STRICT_THRESHOLD:
            name = label_map.get(label_id, "Unknown")
            # Convert distance score to a visual confidence %
            confidence = round(max(0, 100 - (score / STRICT_THRESHOLD * 100)), 1)
            color = (0, 255, 0) # Green for Recognized
            print(f"  [MATCH] {name} | Score: {score:.1f}")
        else:
            name = "Unknown Intruder"
            confidence = 0
            color = (0, 0, 255) # Red for Unknown
            print(f"  [REJECTED] Stranger | Score: {score:.1f}")

        # Draw UI
        label_text = f"{name} {confidence}%" if confidence > 0 else name
        cv2.rectangle(frame, (x, y), (x+w, y+h), color, 2)
        cv2.rectangle(frame, (x, y-35), (x+w, y), color, -1) # Header box
        cv2.putText(frame, label_text, (x+5, y-10), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        
    return frame

# ── 4. Run Live Camera ──────────────────────────────────────
cap = cv2.VideoCapture(0)
print(f"\n[LIVE] Recognition Active. Current Strictness: {STRICT_THRESHOLD}")
print("Watch the console scores to calibrate.")

while True:
    ret, frame = cap.read()
    if not ret: break
    
    output = process_frame(frame)
    cv2.imshow("Security Monitor", output)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

[SYSTEM] Loading face database...
  [LOADED] Ahmad Imad (ID: 0)
  [LOADED] Ahmed Ashraf (ID: 1)
  [LOADED] Ahmed Jaber (ID: 2)
  [LOADED] Ahmed Kamel (ID: 3)


AttributeError: module 'cv2.face' has no attribute 'LBPHFaceRecognizer_create'

In [17]:
pip uninstall dlib face_recognition -y

Found existing installation: dlib 20.0.1
Uninstalling dlib-20.0.1:
  Successfully uninstalled dlib-20.0.1
Found existing installation: face-recognition 1.3.0
Uninstalling face-recognition-1.3.0:
  Successfully uninstalled face-recognition-1.3.0
Note: you may need to restart the kernel to use updated packages.


In [27]:
%pip uninstall dlib face_recognition -y
%pip install deepface tf-keras opencv-python

Note: you may need to restart the kernel to use updated packages.


   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 11.5 MB/s  0:00:00
   ---------------------------------------- 0.0/1.9 MB ? eta -:--:--
   ---------------------------------------- 1.9/1.9 MB 20.7 MB/s  0:00:00

   ----------------------------------------  0/14 [python-dotenv]
   ----------------------------------------  0/14 [python-dotenv]
   ----------------------------------------  0/14 [python-dotenv]
   -------- -------------------------------  3/14 [gunicorn]
   -------- -------------------------------  3/14 [gunicorn]
   -------- -------------------------------  3/14 [gunicorn]
   -------- -------------------------------  3/14 [gunicorn]
   -------- -------------------------------  3/14 [gunicorn]
   -------- -------------------------------  3/14 [gunicorn]
   -------- -------------------------------  3/14 [gunicorn]
   -------- -------------------------------  3/14 [gunicorn]
   -------- ---------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [30]:
# --- 1. SETUP & DOWNLOAD MODELS ---
import os
import urllib.request
import cv2
import numpy as np

MODELS_DIR = "models"
os.makedirs(MODELS_DIR, exist_ok=True)

# UPDATED URLs (Using 'main' instead of 'master')
url_det = "https://raw.githubusercontent.com/opencv/opencv_zoo/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx"
url_rec = "https://raw.githubusercontent.com/opencv/opencv_zoo/main/models/face_recognition_sface/face_recognition_sface_2021dec.onnx"

det_model_path = os.path.join(MODELS_DIR, "face_detection_yunet.onnx")
rec_model_path = os.path.join(MODELS_DIR, "face_recognition_sface.onnx")

def download_model(url, path):
    if not os.path.exists(path):
        print(f"[INFO] Downloading {os.path.basename(path)}... this might take a minute.")
        try:
            urllib.request.urlretrieve(url, path)
            print("[SUCCESS] Download complete!")
        except Exception as e:
            print(f"[ERROR] Failed to download automatically: {e}")

download_model(url_det, det_model_path)
download_model(url_rec, rec_model_path)

[INFO] Downloading face_detection_yunet.onnx... this might take a minute.
[SUCCESS] Download complete!
[INFO] Downloading face_recognition_sface.onnx... this might take a minute.
[SUCCESS] Download complete!


In [ ]:
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu
%pip install facenet-pytorch opencv-python

Looking in indexes: https://download.pytorch.org/whl/cpu
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


INFO: pip is looking at multiple versions of opencv-python to determine which version is compatible with other requirements. This could take a while.
   ---------------------------------------- 0.0/1.9 MB ? eta -:--:--
   --------------------------------- ------ 1.6/1.9 MB 11.9 MB/s eta 0:00:01
   ---------------------------------------- 1.9/1.9 MB 13.0 MB/s  0:00:00
   ---------------------------------------- 0.0/15.8 MB ? eta -:--:--
   --------------------- ------------------ 8.4/15.8 MB 40.0 MB/s eta 0:00:01
   ------------------------------------- -- 14.9/15.8 MB 37.6 MB/s eta 0:00:01
   ---------------------------------------- 15.8/15.8 MB 26.9 MB/s  0:00:00
   ---------------------------------------- 0.0/2.6 MB ? eta -:--:--
   ---------------------------------------- 2.6/2.6 MB 37.4 MB/s  0:00:00
   ---------------------------------------- 0.0/198.6 MB ? eta -:--:--
   -- ------------------------------------- 10.2/198.6 MB 53.0 MB/s eta 0:00:04
   -- ---------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


: 

In [7]:
import cv2
import os
import numpy as np
import torch
from facenet_pytorch import MTCNN, InceptionResnetV1

# ==========================================
# 1. INITIALIZE PYTORCH MODELS
# ==========================================
print("[INFO] Booting PyTorch FaceNet...")
print("[INFO] (If this is your first time running this, it will take ~30s to download the model weights)")

device = torch.device('cpu')
# We need two detectors: one strictly for taking 1 face during encoding, one for multiple faces on camera
mtcnn_encode = MTCNN(keep_all=False, device=device, min_face_size=60)
mtcnn_live   = MTCNN(keep_all=True, device=device, min_face_size=60)
# The Recognition Model
resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

# ==========================================
# ==========================================
# 2. ENCODE YOUR HACKATHON GROUP (OPTIMIZED)
# ==========================================
PROJECT_DIR  = r"d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition"
SAMPLES_DIR  = os.path.join(PROJECT_DIR, "live_samples")

known_embeddings = []
known_names = []

print("\n[INFO] Scanning face database...")

# This disables training memory, making it MUCH faster on CPU
with torch.no_grad(): 
    for person_name in os.listdir(SAMPLES_DIR):
        person_dir = os.path.join(SAMPLES_DIR, person_name)
        if not os.path.isdir(person_dir): continue

        # HACKATHON TRICK: Only grab the first 10 images, ignore the hundreds of augmented ones
        all_images = [f for f in os.listdir(person_dir) if f.endswith(('.jpg', '.png'))]
        selected_images = all_images[:10] 
        
        print(f"  -> Encoding {len(selected_images)} images for '{person_name}'...")

        for img_name in selected_images:
            img_path = os.path.join(person_dir, img_name)
            img = cv2.imread(img_path)
            if img is None: continue
            
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            # Detect face
            face_tensor = mtcnn_encode(img_rgb)
            
            if face_tensor is not None:
                # Extract embedding
                emb = resnet(face_tensor.unsqueeze(0)).detach().numpy().flatten()
                known_embeddings.append(emb)
                known_names.append(person_name)

print(f"[SUCCESS] {len(known_names)} face samples loaded into memory. Starting camera now...")
# ==========================================
# 3. LIVE CAMERA RECOGNITION
# ==========================================
cap = cv2.VideoCapture(0)
print("\n[LIVE] Camera Active. Press 'Q' to quit.")

# HACKATHON CALIBRATION:
# FaceNet uses Euclidean Distance. 
# Lower is better. < 0.8 is usually a match. > 0.9 is usually a stranger.
STRICTNESS_THRESHOLD = 0.85 

while True:
    ret, frame = cap.read()
    if not ret: break

    img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    # Detect all faces in the camera
    boxes, _ = mtcnn_live.detect(img_rgb)
    face_tensors = mtcnn_live(img_rgb)
    
    if boxes is not None and face_tensors is not None:
        # Get embeddings for all detected faces
        embeddings = resnet(face_tensors).detach().numpy()
        
        for i, box in enumerate(boxes):
            box = box.astype(int)
            emb = embeddings[i]
            
            min_dist = float('inf')
            name = "Unknown"
            
            # Compare with database
            for j, known_emb in enumerate(known_embeddings):
                # Euclidean distance formula
                dist = np.linalg.norm(emb - known_emb)
                if dist < min_dist:
                    min_dist = dist
                    
            # Check against your Gate
            if min_dist < STRICTNESS_THRESHOLD:
                # Find the exact name of the best match
                best_idx = np.argmin([np.linalg.norm(emb - k_emb) for k_emb in known_embeddings])
                name = known_names[best_idx]
                color = (0, 255, 0) # Green
                label = f"{name} ({min_dist:.2f})"
            else:
                name = "Unknown Intruder"
                color = (0, 0, 255) # Red
                label = f"Unknown ({min_dist:.2f})"
                
            # Draw UI
            cv2.rectangle(frame, (box[0], box[1]), (box[2], box[3]), color, 2)
            cv2.putText(frame, label, (box[0], box[1] - 10), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    cv2.imshow("FaceNet PyTorch Security", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'): 
        break

cap.release()
cv2.destroyAllWindows()
print("[INFO] System Offline.")

: 

In [6]:
import cv2
import face_recognition
import os
import numpy as np

# --- SETTINGS ---
PROJECT_DIR = r"d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition"
SAMPLES_DIR = os.path.join(PROJECT_DIR, "live_samples")
# 0.6 is standard; 0.5 is very strict (better for "Unknown" rejection)
TOLERANCE = 0.5 

known_face_encodings = []
known_face_names = []

# --- 1. ENCODING PHASE ---
print("[INFO] Encoding faces... This takes a moment.")
for person_name in os.listdir(SAMPLES_DIR):
    person_dir = os.path.join(SAMPLES_DIR, person_name)
    if not os.path.isdir(person_dir): continue

    for img_name in os.listdir(person_dir):
        if img_name.endswith((".jpg", ".png")):
            img_path = os.path.join(person_dir, img_name)
            image = face_recognition.load_image_file(img_path)
            
            # Get the 128-d facial embedding
            encodings = face_recognition.face_encodings(image)
            
            if len(encodings) > 0:
                known_face_encodings.append(encodings[0])
                known_face_names.append(person_name)

print(f"[INFO] Successfully encoded {len(known_face_names)} faces.")

# --- 2. RECOGNITION PHASE ---
video_capture = cv2.VideoCapture(0)

while True:
    ret, frame = video_capture.read()
    if not ret: break

    # Resize frame to 1/4 size for much faster processing
    small_frame = cv2.resize(frame, (0, 0), fx=0.25, fy=0.25)
    # Convert BGR (OpenCV) to RGB (face_recognition)
    rgb_small_frame = cv2.cvtColor(small_frame, cv2.COLOR_BGR2RGB)

    # Find all faces and encodings in the current frame
    face_locations = face_recognition.face_locations(rgb_small_frame)
    face_encodings = face_recognition.face_encodings(rgb_small_frame, face_locations)

    for (top, right, bottom, left), face_encoding in zip(face_locations, face_encodings):
        # Compare current face to our database
        matches = face_recognition.compare_faces(known_face_encodings, face_encoding, tolerance=TOLERANCE)
        name = "Unknown"

        # Calculate "distance" (lower is more similar)
        face_distances = face_recognition.face_distance(known_face_encodings, face_encoding)
        if len(face_distances) > 0:
            best_match_index = np.argmin(face_distances)
            if matches[best_match_index]:
                name = known_face_names[best_match_index]

        # Scale back up (since we processed at 1/4 size)
        top *= 4; right *= 4; bottom *= 4; left *= 4

        # Draw UI
        color = (0, 255, 0) if name != "Unknown" else (0, 0, 255)
        cv2.rectangle(frame, (left, top), (right, bottom), color, 2)
        cv2.rectangle(frame, (left, bottom - 35), (right, bottom), color, cv2.FILLED)
        cv2.putText(frame, name, (left + 6, bottom - 6), cv2.FONT_HERSHEY_DUPLEX, 0.8, (255, 255, 255), 1)

    cv2.imshow('Deep Learning Face Recognition', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

video_capture.release()
cv2.destroyAllWindows()

ImportError: DLL load failed while importing _dlib_pybind11: A dynamic link library (DLL) initialization routine failed.

In [1]:
%pip uninstall dlib face_recognition -y

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install face_recognition

  Using cached face_recognition-1.3.0-py2.py3-none-any.whl.metadata (21 kB)
  Using cached dlib-20.0.1-cp311-cp311-win_amd64.whl
Using cached face_recognition-1.3.0-py2.py3-none-any.whl (15 kB)

   -------------------- ------------------- 1/2 [face_recognition]
   ---------------------------------------- 2/2 [face_recognition]

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# ==========================================
# 2. ENCODE YOUR HACKATHON GROUP (OPTIMIZED)
# ==========================================
PROJECT_DIR  = r"d:\Self_Study\DEBI\DEBI-Hackathion-Face-Recognition"
SAMPLES_DIR  = os.path.join(PROJECT_DIR, "live_samples")

known_embeddings = []
known_names = []

print("\n[INFO] Scanning face database...")

# This disables training memory, making it MUCH faster on CPU
with torch.no_grad(): 
    for person_name in os.listdir(SAMPLES_DIR):
        person_dir = os.path.join(SAMPLES_DIR, person_name)
        if not os.path.isdir(person_dir): continue

        # HACKATHON TRICK: Only grab the first 10 images, ignore the hundreds of augmented ones
        all_images = [f for f in os.listdir(person_dir) if f.endswith(('.jpg', '.png'))]
        selected_images = all_images[:10] 
        
        print(f"  -> Encoding {len(selected_images)} images for '{person_name}'...")

        for img_name in selected_images:
            img_path = os.path.join(person_dir, img_name)
            img = cv2.imread(img_path)
            if img is None: continue
            
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            # Detect face
            face_tensor = mtcnn_encode(img_rgb)
            
            if face_tensor is not None:
                # Extract embedding
                emb = resnet(face_tensor.unsqueeze(0)).detach().numpy().flatten()
                known_embeddings.append(emb)
                known_names.append(person_name)

print(f"[SUCCESS] {len(known_names)} face samples loaded into memory. Starting camera now...")

SyntaxError: invalid syntax (1244603717.py, line 2)

In [9]:
import json
import os
import cv2
import numpy as np
import face_recognition

ImportError: DLL load failed while importing _dlib_pybind11: A dynamic link library (DLL) initialization routine failed.

In [3]:
#mount the drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
"""
seed_db.py — Batch Face Enrollment from Static Photos
Scans the known_faces/ folder, extracts face embeddings from each image,
and saves all {name, embedding} entries into database.json.

Usage:
    1. Place one clear photo per person in known_faces/
       (e.g. ahmed.jpg, sara.png)
    2. Run:  python seed_db.py
"""

import json
import os
import face_recognition

DB_PATH = os.environ.get("DB_PATH", "/content/drive/MyDrive/Hackathon/database.json")
KNOWN_FACES_DIR = "/content/drive/MyDrive/Hackathon/known_faces"
# /content/drive/MyDrive/Hackathon/known_faces
SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png"}


def load_database(path: str) -> dict:
    """Load the existing face database from disk, or create a fresh one."""
    if os.path.exists(path):
        with open(path, "r") as f:
            return json.load(f)
    return {"persons": []}


def save_database(path: str, db: dict) -> None:
    """Persist the face database to disk."""
    with open(path, "w") as f:
        json.dump(db, f, indent=4)


def seed():
    """Scan known_faces/ and enroll every detected face into the database."""
    print("=" * 50)
    print("  DATABASE SEEDING FROM STATIC PHOTOS")
    print("=" * 50)
    print(f"[INFO] DB_PATH = {DB_PATH}")

    if not os.path.isdir(KNOWN_FACES_DIR):
        print(f"[ERROR] Directory '{KNOWN_FACES_DIR}/' not found. "
              "Please create it and add photos.")
        return

    # Collect image files
    image_files = [
        f for f in sorted(os.listdir(KNOWN_FACES_DIR))
        if os.path.splitext(f)[1].lower() in SUPPORTED_EXTENSIONS
    ]

    if not image_files:
        print(f"[WARNING] No image files found in '{KNOWN_FACES_DIR}/'. "
              "Add .jpg, .jpeg, or .png files and try again.")
        return

    print(f"[INFO] Found {len(image_files)} image(s) in '{KNOWN_FACES_DIR}/'.\n")

    db = load_database(DB_PATH)
    enrolled = 0
    skipped = 0

    for filename in image_files:
        name = os.path.splitext(filename)[0]
        filepath = os.path.join(KNOWN_FACES_DIR, filename)

        print(f"  Processing: {filename} → \"{name}\"")

        # Load image and extract face encodings
        image = face_recognition.load_image_file(filepath)
        encodings = face_recognition.face_encodings(image)

        if len(encodings) == 0:
            print(f"  [WARNING] No face detected in '{filename}'. Skipping.\n")
            skipped += 1
            continue

        if len(encodings) > 1:
            print(f"  [WARNING] Multiple faces in '{filename}'. "
                  "Using the first detected face.")

        embedding = encodings[0].tolist()

        db["persons"].append({
            "name": name,
            "embedding": embedding
        })

        enrolled += 1
        print(f"  [SUCCESS] '{name}' enrolled successfully!\n")

    save_database(DB_PATH, db)

    print("-" * 50)
    print(f"  Done!  Enrolled: {enrolled}  |  Skipped: {skipped}")
    print(f"  Total persons in database: {len(db['persons'])}")
    print("-" * 50)


if __name__ == "__main__":
    seed()


  DATABASE SEEDING FROM STATIC PHOTOS
[INFO] DB_PATH = /content/drive/MyDrive/Hackathon/database.json
[INFO] Found 3 image(s) in '/content/drive/MyDrive/Hackathon/known_faces/'.

  Processing: Ahmed Ashraf.jpeg → "Ahmed Ashraf"
  [SUCCESS] 'Ahmed Ashraf' enrolled successfully!

  Processing: Ahmed Jaber.JPG → "Ahmed Jaber"
  [SUCCESS] 'Ahmed Jaber' enrolled successfully!

  Processing: Ahmed Kamel.jpeg → "Ahmed Kamel"
  [SUCCESS] 'Ahmed Kamel' enrolled successfully!

--------------------------------------------------
  Done!  Enrolled: 3  |  Skipped: 0
  Total persons in database: 3
--------------------------------------------------


In [7]:
import base64
import io
import json
import os

import cv2
import numpy as np
import face_recognition
from PIL import Image

# ── Configuration ──────────────────────────────────────────────────────────
DB_PATH = os.environ.get("DB_PATH", "/kaggle/working/database.json")
TOLERANCE = 0.5
RESIZE_SCALE = 0.25  # Process frames at 25% resolution for speed

# ── Load enrolled faces at startup ─────────────────────────────────────────
print("=" * 50)
print("  FACE RECOGNITION ENGINE — JUPYTER MODE")
print("=" * 50)
print(f"[INFO] DB_PATH = {DB_PATH}")


def _load_database(path: str) -> dict:
    """Load the face database from disk, or return an empty one."""
    if os.path.exists(path):
        with open(path, "r") as f:
            return json.load(f)
    return {"persons": []}


_db = _load_database(DB_PATH)
known_names = [p["name"] for p in _db["persons"]]
known_encodings = [np.array(p["embedding"]) for p in _db["persons"]]

print(f"[INFO] Loaded {len(known_names)} enrolled face(s).")
if not known_names:
    print("[WARNING] Database is empty. All faces will be labelled 'Unknown'.")
print("[INFO] Ready to process frames.\n")


# ── Helper: encode a BGR numpy frame → base64 JPEG ────────────────────────
def _encode_frame(frame: np.ndarray) -> str:
    """Convert a BGR numpy frame to a base64-encoded JPEG string."""
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    pil_out = Image.fromarray(rgb)
    buf = io.BytesIO()
    pil_out.save(buf, format="JPEG", quality=80)
    return base64.b64encode(buf.getvalue()).decode("utf-8")


# ── Core: process a single frame ──────────────────────────────────────────
def process_frame(b64_image: str) -> str:
    """
    Receive a base64 JPEG frame from the browser, run face recognition,
    and return the annotated frame as a base64 JPEG string.

    If no face is detected the original frame is returned unmodified.
    """
    # ── Decode base64 → numpy BGR array ──
    img_bytes = base64.b64decode(b64_image)
    pil_img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
    frame = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)

    # ── Resize to 25 % for faster detection ──
    small = cv2.resize(frame, (0, 0), fx=RESIZE_SCALE, fy=RESIZE_SCALE)
    rgb_small = cv2.cvtColor(small, cv2.COLOR_BGR2RGB)

    face_locs = face_recognition.face_locations(rgb_small)
    face_encs = face_recognition.face_encodings(rgb_small, face_locs)

    # No faces → return original frame unmodified
    if not face_locs:
        return _encode_frame(frame)

    # ── Match each face & draw annotations ──
    scale = int(1 / RESIZE_SCALE)

    for (top, right, bottom, left), enc in zip(face_locs, face_encs):
        name = "Unknown"
        color = (0, 0, 255)  # Red for unknown

        if known_encodings:
            matches = face_recognition.compare_faces(
                known_encodings, enc, tolerance=TOLERANCE
            )
            distances = face_recognition.face_distance(known_encodings, enc)
            best_idx = int(np.argmin(distances))
            if matches[best_idx]:
                name = known_names[best_idx]
                color = (0, 255, 0)  # Green for recognized

        # Scale bounding box back to full resolution
        top *= scale
        right *= scale
        bottom *= scale
        left *= scale

        # Bounding box
        cv2.rectangle(frame, (left, top), (right, bottom), color, 2)

        # Label background + text
        label_y = top - 10 if top - 10 > 10 else top + 20
        cv2.rectangle(
            frame, (left, label_y - 18), (right, label_y + 4), color, cv2.FILLED
        )
        cv2.putText(
            frame, name, (left + 4, label_y),
            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1,
        )

    return _encode_frame(frame)


  FACE RECOGNITION ENGINE — JUPYTER MODE
[INFO] DB_PATH = /kaggle/working/database.json
[INFO] Loaded 0 enrolled face(s).
[WARNING] Database is empty. All faces will be labelled 'Unknown'.
[INFO] Ready to process frames.



In [8]:
# ── Cell 1: Face Recognition Engine ─────────────────────────────────────
# Loads enrolled embeddings and defines process_frame(b64) → b64

import base64
import io
import json
import os

import cv2
import numpy as np
import face_recognition
from PIL import Image

# ── Configuration ──
DB_PATH = os.environ.get("DB_PATH", "/kaggle/working/database.json")
TOLERANCE = 0.5
RESIZE_SCALE = 0.25

# ── Load enrolled faces ──
print("=" * 50)
print("  FACE RECOGNITION ENGINE — JUPYTER MODE")
print("=" * 50)
print(f"[INFO] DB_PATH = {DB_PATH}")

def _load_database(path):
    if os.path.exists(path):
        with open(path, "r") as f:
            return json.load(f)
    return {"persons": []}

_db = _load_database(DB_PATH)
known_names = [p["name"] for p in _db["persons"]]
known_encodings = [np.array(p["embedding"]) for p in _db["persons"]]

print(f"[INFO] Loaded {len(known_names)} enrolled face(s).")
if not known_names:
    print("[WARNING] Database is empty. All faces will be labelled 'Unknown'.")
print("[INFO] Ready to process frames.\n")


def _encode_frame(frame):
    """BGR numpy → base64 JPEG string."""
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    pil_out = Image.fromarray(rgb)
    buf = io.BytesIO()
    pil_out.save(buf, format="JPEG", quality=80)
    return base64.b64encode(buf.getvalue()).decode("utf-8")


def process_frame(b64_image):
    """
    base64 JPEG in → face recognition → annotated base64 JPEG out.
    Returns the original frame unmodified if no faces are detected.
    """
    img_bytes = base64.b64decode(b64_image)
    pil_img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
    frame = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)

    small = cv2.resize(frame, (0, 0), fx=RESIZE_SCALE, fy=RESIZE_SCALE)
    rgb_small = cv2.cvtColor(small, cv2.COLOR_BGR2RGB)

    face_locs = face_recognition.face_locations(rgb_small)
    face_encs = face_recognition.face_encodings(rgb_small, face_locs)

    if not face_locs:
        return _encode_frame(frame)

    scale = int(1 / RESIZE_SCALE)

    for (top, right, bottom, left), enc in zip(face_locs, face_encs):
        name = "Unknown"
        color = (0, 0, 255)

        if known_encodings:
            matches = face_recognition.compare_faces(
                known_encodings, enc, tolerance=TOLERANCE
            )
            distances = face_recognition.face_distance(known_encodings, enc)
            best_idx = int(np.argmin(distances))
            if matches[best_idx]:
                name = known_names[best_idx]
                color = (0, 255, 0)

        top *= scale
        right *= scale
        bottom *= scale
        left *= scale

        cv2.rectangle(frame, (left, top), (right, bottom), color, 2)
        label_y = top - 10 if top - 10 > 10 else top + 20
        cv2.rectangle(frame, (left, label_y - 18), (right, label_y + 4), color, cv2.FILLED)
        cv2.putText(frame, name, (left + 4, label_y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)

    return _encode_frame(frame)

print("[INFO] process_frame() is ready.")


  FACE RECOGNITION ENGINE — JUPYTER MODE
[INFO] DB_PATH = /kaggle/working/database.json
[INFO] Loaded 0 enrolled face(s).
[WARNING] Database is empty. All faces will be labelled 'Unknown'.
[INFO] Ready to process frames.

[INFO] process_frame() is ready.


In [10]:
# ── Cell 2: Browser Camera + Live Recognition ───────────────────────────
from IPython.display import display, HTML

display(HTML("""
<div id="recognition-app" style="display:flex; flex-direction:column; align-items:center;
     gap:12px; font-family:system-ui,sans-serif; padding:20px;">

    <h3 style="margin:0;">📹 Live Face Recognition</h3>

    <!-- Live camera preview -->
    <video id="webcam" autoplay playsinline muted
           width="640" height="480"
           style="border:2px solid #555; border-radius:8px; background:#000;">
    </video>

    <!-- Controls -->
    <div style="display:flex; gap:10px;">
        <button id="start-btn"
                style="padding:10px 28px; font-size:15px; cursor:pointer;
                       background:#2ecc71; color:#fff; border:none; border-radius:8px;">
            ▶ Start Recognition
        </button>
        <button id="stop-btn"
                style="padding:10px 28px; font-size:15px; cursor:pointer;
                       background:#e74c3c; color:#fff; border:none; border-radius:8px;">
            ⏹ Stop
        </button>
    </div>

    <!-- Recognition output -->
    <h4 style="margin:0;">🔍 Recognition Output</h4>
    <img id="output-img" width="640" height="480"
         style="border:2px solid #2ecc71; border-radius:8px; background:#111;" />

    <!-- Status log -->
    <div id="status-log"
         style="width:640px; max-height:180px; overflow-y:auto;
                background:#1a1a2e; color:#eee; font-family:monospace; font-size:13px;
                padding:12px; border-radius:8px; white-space:pre-line;">
        Waiting for user action...
    </div>
</div>

<script>
(function() {
    // ── DOM refs ──
    var video      = document.getElementById('webcam');
    var outputImg  = document.getElementById('output-img');
    var startBtn   = document.getElementById('start-btn');
    var stopBtn    = document.getElementById('stop-btn');
    var statusLog  = document.getElementById('status-log');
    var canvas     = document.createElement('canvas');
    var ctx        = canvas.getContext('2d');

    var intervalId  = null;
    var processing  = false;
    var stream      = null;

    // ── Logging helper ──
    function log(msg) {
        var ts = new Date().toLocaleTimeString();
        statusLog.textContent = '[' + ts + '] ' + msg + '\\n' + statusLog.textContent;
    }

    // ── Check if a frame is non-black ──
    // Samples a 50x50 region from the center and checks avg pixel > 10
    function isFrameReal() {
        if (video.videoWidth === 0 || video.videoHeight === 0) return false;
        canvas.width  = 50;
        canvas.height = 50;
        var sx = Math.floor(video.videoWidth / 2) - 25;
        var sy = Math.floor(video.videoHeight / 2) - 25;
        ctx.drawImage(video, sx, sy, 50, 50, 0, 0, 50, 50);
        var data = ctx.getImageData(0, 0, 50, 50).data;
        var sum  = 0;
        for (var i = 0; i < data.length; i += 4) {
            sum += data[i] + data[i+1] + data[i+2];  // R + G + B
        }
        var avg = sum / (50 * 50 * 3);
        return avg > 10;
    }

    // ── Wait for video readyState >= 2 ──
    function waitForVideoReady() {
        return new Promise(function(resolve) {
            if (video.readyState >= 2) {
                resolve();
            } else {
                video.addEventListener('loadeddata', function onLoaded() {
                    video.removeEventListener('loadeddata', onLoaded);
                    resolve();
                });
            }
        });
    }

    // ── Warm-up: wait for 5 consecutive non-black frames ──
    function warmUp() {
        return new Promise(function(resolve) {
            var goodCount = 0;
            var attempts  = 0;
            var maxAttempts = 100;  // 10 seconds max at 100ms interval

            log('🔄 Warm-up: waiting for real frames (non-black)...');

            var checkInterval = setInterval(function() {
                attempts++;
                if (isFrameReal()) {
                    goodCount++;
                    log('  ✓ Non-black frame ' + goodCount + '/5 detected');
                } else {
                    goodCount = 0;  // reset — need 5 consecutive
                }

                if (goodCount >= 5) {
                    clearInterval(checkInterval);
                    log('✅ Warm-up complete! Camera is delivering real frames.');
                    resolve(true);
                } else if (attempts >= maxAttempts) {
                    clearInterval(checkInterval);
                    log('⚠️ Warm-up timed out after 10s. Frames may still be black.');
                    resolve(false);
                }
            }, 100);
        });
    }

    // ── Capture one frame and send to Python kernel ──
    function captureAndSend() {
        if (processing) return;
        if (video.videoWidth === 0) return;

        canvas.width  = video.videoWidth;
        canvas.height = video.videoHeight;
        ctx.drawImage(video, 0, 0);

        var dataUrl = canvas.toDataURL('image/jpeg', 0.8);
        var b64     = dataUrl.split(',')[1];

        processing = true;
        log('📤 Frame sent to Python...');

        var kernel = (typeof Jupyter !== 'undefined' && Jupyter.notebook)
                     ? Jupyter.notebook.kernel
                     : IPython.notebook.kernel;

        var code = '_b64_input = """' + b64 + '"""' + '\\n' + 'print(process_frame(_b64_input))';

        kernel.execute(code, {
            iopub: {
                output: function(msg) {
                    if (msg.msg_type === 'stream' && msg.content.name === 'stdout') {
                        var result = msg.content.text.trim();
                        if (result.length > 100) {
                            outputImg.src = 'data:image/jpeg;base64,' + result;
                            log('✅ Result received, displaying.');
                        }
                    }
                    processing = false;
                }
            }
        });

        // Safety timeout
        setTimeout(function() { processing = false; }, 8000);
    }

    // ── START button ──
    startBtn.addEventListener('click', async function() {
        if (intervalId) {
            log('⚠️ Already running.');
            return;
        }

        try {
            // Step 1: Request camera
            log('📷 Requesting camera permission...');
            stream = await navigator.mediaDevices.getUserMedia({
                video: { width: 640, height: 480, facingMode: 'user' }
            });

            video.srcObject = stream;
            video.play();
            log('📹 Camera stream started, warming up...');

            // Step 2: Wait for video element to have data
            await waitForVideoReady();
            log('📺 Video element ready (readyState=' + video.readyState + ')');

            // Step 3: Explicit 2-second delay for camera auto-exposure
            log('⏳ Waiting 2 seconds for camera auto-exposure...');
            await new Promise(function(r) { setTimeout(r, 2000); });

            // Step 4: Warm-up loop — wait for 5 non-black frames
            var warmedUp = await warmUp();

            if (!warmedUp) {
                log('⚠️ Camera may not be working. Trying recognition anyway...');
            }

            // Step 5: Start the recognition loop
            log('🟢 Camera ready, starting recognition loop (every 200ms)...');
            intervalId = setInterval(captureAndSend, 200);

        } catch(err) {
            log('❌ Camera error: ' + err.message);
            log('💡 Try the Upload method in the next cell instead.');
        }
    });

    // ── STOP button ──
    stopBtn.addEventListener('click', function() {
        if (intervalId) {
            clearInterval(intervalId);
            intervalId = null;
        }
        processing = false;

        // Stop camera stream
        if (stream) {
            stream.getTracks().forEach(function(t) { t.stop(); });
            stream = null;
        }
        video.srcObject = null;

        log('⏹ Stopped. Camera released.');
    });

    log('Ready. Press ▶ Start Recognition to begin.');
})();
</script>
"""))


In [24]:
# ── Cell 3: Upload Photo → Recognition (guaranteed to work) ─────────────
import ipywidgets as widgets

print("📤 Upload a photo to run face recognition on it.\n")

upload = widgets.FileUpload(accept='.jpg,.jpeg,.png', multiple=True)
run_btn = widgets.Button(
    description='🔍 Run Recognition',
    button_style='success',
    layout=widgets.Layout(width='200px', height='40px')
)
output_area = widgets.Output()

def on_run_clicked(btn):
    output_area.clear_output()
    with output_area:
        if not upload.value:
            print("⚠️  Please upload at least one photo first.")
            return
        
        for file_info in upload.value:
            name = file_info.name
            content = file_info.content
            
            print(f"Processing: {name}")
            b64_input = base64.b64encode(content).decode("utf-8")
            b64_result = process_frame(b64_input)
            
            display(HTML(
                f'<p style="font-weight:bold; font-family:sans-serif;">{name}:</p>'
                f'<img src="data:image/jpeg;base64,{b64_result}" '
                f'style="max-width:640px; border:2px solid #2ecc71; '
                f'border-radius:8px; margin-bottom:20px;" />'
            ))
            print()

run_btn.on_click(on_run_clicked)

display(widgets.VBox([
    upload,
    run_btn,
    output_area
]))


📤 Upload a photo to run face recognition on it.

